# 07 — Fold 3 Final Production Simulation

**This notebook is run exactly once. It is never rerun. The moment Fold 3 results are
visible, the pipeline is locked regardless of what they show.**

Fold 3 is not a model tuning step. It is a production simulation: does the full
pipeline — routing, forecasting, uncertainty quantification, and inventory
simulation — behave consistently on unseen data?

**Inputs (Fold 2 lock-ins, reused verbatim, not re-derived):**
- `winner_decision_fold2.pkl` — 06b, production model identity
- `regime_thresholds.pkl` — 06c, ADI/CV² thresholds (1.32, 0.49)
- `conformal_residuals_fold2.pkl` — 06e, locked per-regime service level / alpha
- `features_train_v2.parquet` (train, ends 2015-01-31) and `features_val_v2.parquet`
  (val, 2015-02-01 -> 2016-01-31) — 04b, two separate files, not one date-split
- `feature_cols_v2.pkl` — 04b
- 06g Sections 7 and 9's cost formulas and parameters (safety stock, review period,
  lead time, `52/N_WEEKS_VAL` annualization) — reused directly in Section 5b, not
  re-derived

**Outputs:**
- `tweedie_optimized_fold3.txt`, `sku_regimes_fold3.parquet`,
  `conformal_residuals_fold3.pkl`, `final_predictions_fold3.parquet`,
  `simulation_results_fold3.parquet`
- `dynamic_cost_sensitivity_fold3.parquet` (Section 5b, mirrors 06g Section 8)
- `static_vs_dynamic_headtohead_fold3.parquet` (Section 5b, mirrors 06g Section 9 —
  this is the app's headline source)

---

## Section 1 — Pre-Flight Checklist

Confirm every locked decision below BEFORE any Fold 3 data is touched. Nothing in
this section re-derives a decision — it only checks that what's on disk still
matches what prior notebooks locked, and defines the Fold 3 train/val split.


In [4]:
# -- 07 Section 1: Pre-Flight Checklist --------------------------------------------------
import numpy as np
import pandas as pd
import pickle
import os
import warnings
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

PROCESSED_DIR    = '../data/processed'
CALIBRATION_DIR  = f'{PROCESSED_DIR}/calibration'
SEGMENTATION_DIR = f'{PROCESSED_DIR}/segmentation'
MODELS_DIR       = f'{PROCESSED_DIR}/models'
PREDICTIONS_DIR  = f'{PROCESSED_DIR}/predictions'
FEATURES_DIR     = f'{PROCESSED_DIR}/features'

FOLD3_VAL_START = pd.Timestamp('2015-02-01')

# -- Locked decisions from pre-Fold 3 work -- confirm, do not re-derive ------------------
# 06d tested a direct 7-day target and asymmetric loss; both failed the adoption gate.
# Neither was adopted -- PRODUCTION_MODEL is the unmodified 06b baseline, saved under 06d's
# output filename (tweedie_optimized_fold2.txt) but architecturally identical to 06b.
PRODUCTION_MODEL  = 'tweedie_baseline'
HYSTERESIS_WEEKS  = 4

# -- Confirm 06b's winner decision is what this notebook assumes ------------------------
winner_path = f'{CALIBRATION_DIR}/winner_decision_fold2.pkl'
assert os.path.exists(winner_path), f'Missing {winner_path} -- 06b winner lock-in not found.'
with open(winner_path, 'rb') as f:
    winner = pickle.load(f)

print('-- 06b winner decision (locked) --')
print(f"  Model:   {winner.get('WINNER_MODEL')}")
print(f"  Variant: {winner.get('WINNER_VARIANT')}")
assert winner.get('WINNER_MODEL') == 'tweedie', (
    f"winner_decision_fold2.pkl says {winner.get('WINNER_MODEL')!r}, not 'tweedie' -- "
    f"PRODUCTION_MODEL assumption above is stale, stop and check 06b before proceeding."
)
# NOTE: winner['WINNER_VARIANT'] is 'raw', not 'suppressed' -- new_plan.md's Immediate-
# Next-Step draft said 'suppressed'; the pkl is the real locked artifact and takes
# precedence. Plan doc update pending confirmation against 06b's own notebook.

# -- Regime thresholds -- locked in 06c, reused verbatim for Fold 3 reclassification ----
thresholds_path = f'{SEGMENTATION_DIR}/regime_thresholds.pkl'
assert os.path.exists(thresholds_path), f'Missing {thresholds_path} -- 06c output not found.'
with open(thresholds_path, 'rb') as f:
    thresholds_meta = pickle.load(f)

# BUG FIXED (v1 of this cell): compared the whole metadata dict (keys adi_threshold/
# cv2_threshold plus counts/fold/train_end) against a plain {'adi', 'cv2'} dict -- always
# false regardless of the real values. Extract the two threshold values explicitly instead.
REGIME_THRESHOLDS = {
    'adi': thresholds_meta['adi_threshold'],
    'cv2': thresholds_meta['cv2_threshold'],
}
assert REGIME_THRESHOLDS == {'adi': 1.32, 'cv2': 0.49}, (
    f'REGIME_THRESHOLDS on disk ({REGIME_THRESHOLDS}) does not match new_plan.md -- '
    f'Section 2 reclassification would use different thresholds than every prior fold.'
)
print(f'Regime thresholds (locked, 06c): {REGIME_THRESHOLDS}')

FOLD2_REGIME_COUNTS = thresholds_meta['regime_counts']
print(f'Fold 2 regime counts (for Fold 2 vs Fold 3 comparison in Section 2): {FOLD2_REGIME_COUNTS}')

# -- Conformal config -- locked in 06e, refit ON Fold 3 data in Section 4, same method/alpha
conformal_path = f'{CALIBRATION_DIR}/conformal_residuals_fold2.pkl'
assert os.path.exists(conformal_path), f'Missing {conformal_path} -- 06e output not found.'
with open(conformal_path, 'rb') as f:
    conformal_fold2 = pickle.load(f)

SERVICE_LEVEL_BY_REGIME = conformal_fold2['default_service_level']
CONFORMAL_ALPHA = {
    regime: round(1 - int(level[1:]) / 100, 2)
    for regime, level in SERVICE_LEVEL_BY_REGIME.items()
}
print(f'Locked service levels by regime (06e): {SERVICE_LEVEL_BY_REGIME}')
print(f'Corresponding conformal alpha by regime: {CONFORMAL_ALPHA}')
print(f'Hysteresis: {HYSTERESIS_WEEKS} weeks (locked, 06c)')

# -- Load Fold 3 train/val -- TWO SEPARATE FILES, not a date-split of one ----------------
# features_train_v2.parquet ends 2015-01-31 and never contains Fold 3's val window.
# features_val_v2.parquet holds 2015-02-01 -> 2016-01-31 instead (confirmed by direct
# inspection: same schema, all 30,490 SKUs, 365 unique dates) -- Fold 3 train/val are
# sourced from two different files by design, not filtered from one shared file.
fold3_train = pd.read_parquet(f'{FEATURES_DIR}/features_train_v2.parquet')
fold3_train['date'] = pd.to_datetime(fold3_train['date'])

fold3_val = pd.read_parquet(f'{FEATURES_DIR}/features_val_v2.parquet')
fold3_val['date'] = pd.to_datetime(fold3_val['date'])

assert set(fold3_val.columns) == set(fold3_train.columns), (
    'features_val_v2.parquet schema does not match features_train_v2.parquet -- cannot '
    'safely apply the same feature_cols/model to both.'
)

print(f"\nFold 3 train: {fold3_train['date'].min().date()} -> {fold3_train['date'].max().date()} "
      f"({fold3_train['date'].nunique():,} days, {len(fold3_train):,} rows)")
print(f"Fold 3 val:   {fold3_val['date'].min().date()} -> {fold3_val['date'].max().date()} "
      f"({fold3_val['date'].nunique():,} days, {len(fold3_val):,} rows)")

# -- Extra safety check beyond the plan's two asserts: since train/val now come from two
# independently-loaded files rather than one date-split, confirm no date appears in both --
# a stale or duplicated file could otherwise leak val dates into train silently. ---------
overlap_dates = set(fold3_train['date'].unique()) & set(fold3_val['date'].unique())
assert not overlap_dates, f'{len(overlap_dates)} date(s) appear in both train and val -- leakage.'

# -- Pre-flight assertions (verbatim from new_plan.md) -----------------------------------
assert fold3_val['date'].min() == pd.Timestamp('2015-02-01'), 'Fold 3 start mismatch'
assert fold3_train['date'].max() < pd.Timestamp('2015-02-01'), 'Leakage detected'
print('\nPre-flight assertions passed.')


-- 06b winner decision (locked) --
  Model:   tweedie
  Variant: raw
Regime thresholds (locked, 06c): {'adi': 1.32, 'cv2': 0.49}
Fold 2 regime counts (for Fold 2 vs Fold 3 comparison in Section 2): {'intermittent': 14268, 'smooth': 8389, 'lumpy': 7003, 'erratic': 830}
Locked service levels by regime (06e): {'smooth': 'q80', 'erratic': 'q80'}
Corresponding conformal alpha by regime: {'smooth': 0.2, 'erratic': 0.2}
Hysteresis: 4 weeks (locked, 06c)

Fold 3 train: 2011-02-02 -> 2015-01-31 (1,460 days, 28,699,814 rows)
Fold 3 val:   2015-02-01 -> 2016-01-31 (365 days, 11,128,850 rows)

Pre-flight assertions passed.


#### Section 1 Findings — Pre-Flight Checklist

**Status: LOCKED**, with one open item flagged below (does not block Section 2).

**Confirmed against actual saved artifacts, not assumed:**
- `winner_decision_fold2.pkl`: `WINNER_MODEL='tweedie'`, `WINNER_VARIANT='raw'`.
- `regime_thresholds.pkl`: `adi=1.32`, `cv2=0.49` — matches every prior fold. Fold 2 regime
  counts captured (`intermittent`: 14,268, `smooth`: 8,389, `lumpy`: 7,003, `erratic`: 830)
  for the Fold 2 vs. Fold 3 stability comparison in Section 2.
- `conformal_residuals_fold2.pkl`: locked service levels `{'smooth': 'q80', 'erratic': 'q80'}`,
  i.e. `alpha=0.20` for both. Lumpy/Intermittent have no conformal wrapper by design (native
  Croston/TSB uncertainty, per Decision 2) — `CONFORMAL_ALPHA` intentionally only covers
  Smooth/Erratic.
- Fold 3 train/val: sourced from two separate files, not a date-split of one —
  `features_train_v2.parquet` (2011-02-02 → 2015-01-31, 28,699,814 rows) and
  `features_val_v2.parquet` (2015-02-01 → 2016-01-31, 11,128,850 rows). Schema match and
  zero date-overlap both confirmed. Both plan-specified pre-flight assertions passed.


## Section 2 — Demand Regime Classification on Fold 3 Training Data

Reclassify all 30,490 SKUs using Fold 3's training window (`fold3_train`), reusing
`compute_adi` / `compute_cv2` / `classify_regime` / `assign_routing` and the locked
thresholds (`adi=1.32`, `cv2=0.49`) verbatim from `06c_sku_audit.ipynb`. No hysteresis
gate is applied — `boundary_flag` only, matching every prior fold (see decision above
Section 2). Compares Fold 3's regime distribution and per-SKU assignments against Fold 2
to flag SKUs that reclassified.

In [2]:
# -- 07 Section 2: Demand Regime Classification on Fold 3 Training Data -----------------
# Reuses compute_adi / compute_cv2 / classify_regime / assign_routing VERBATIM from
# 06c_sku_audit.ipynb cell 7. Same boundary_flag-only approach as every prior fold —
# 06c never implemented a hysteresis gate in code (confirmed by reading that cell
# directly), so Fold 3 does not introduce one either. This keeps Fold 2 vs Fold 3
# regime-count and reclassification comparisons apples-to-apples.

ADI_THRESHOLD = REGIME_THRESHOLDS['adi']   # 1.32, locked in Section 1
CV2_THRESHOLD = REGIME_THRESHOLDS['cv2']   # 0.49, locked in Section 1

def compute_adi(series: pd.Series) -> float:
    """Total periods / nonzero periods. Higher = more intermittent."""
    n_total   = len(series)
    n_nonzero = (series > 0).sum()
    return n_total / n_nonzero if n_nonzero > 0 else np.inf

def compute_cv2(series: pd.Series) -> float:
    """CV² of nonzero demand observations only."""
    nonzero = series[series > 0]
    if len(nonzero) < 2:
        return np.inf
    return (nonzero.std() / nonzero.mean()) ** 2

def classify_regime(adi: float, cv2: float) -> str:
    """Syntetos-Boylan (2005) four-regime classification."""
    if   adi < ADI_THRESHOLD and cv2 < CV2_THRESHOLD:  return 'smooth'
    elif adi < ADI_THRESHOLD and cv2 >= CV2_THRESHOLD: return 'erratic'
    elif adi >= ADI_THRESHOLD and cv2 < CV2_THRESHOLD: return 'intermittent'
    else:                                               return 'lumpy'

def assign_routing(regime: str) -> str:
    return {
        'smooth':       'tweedie',
        'erratic':      'tweedie',
        'intermittent': 'croston',
        'lumpy':        'policy',
    }[regime]

print('Aggregating Fold 3 training data to weekly frequency for ADI/CV² computation...')
print('(Decision frequency = weekly, so classification frequency = weekly)')
print()

fold3_train_weekly = (
    fold3_train
    .groupby(['id', pd.Grouper(key='date', freq='W')])['units_sold']
    .sum()
    .reset_index()
)

print(f'Weekly training rows: {len(fold3_train_weekly):,}')
print(f'Weeks per SKU (median): '
      f'{fold3_train_weekly.groupby("id")["units_sold"].count().median():.0f}')
print()

sku_regimes_fold3 = (
    fold3_train_weekly
    .groupby('id')['units_sold']
    .agg(
        adi = compute_adi,
        cv2 = compute_cv2,
    )
    .reset_index()
)

# Cap inf values for SKUs with no nonzero demand in training -- same cap as 06c
sku_regimes_fold3['adi'] = sku_regimes_fold3['adi'].replace(np.inf, 999)
sku_regimes_fold3['cv2'] = sku_regimes_fold3['cv2'].replace(np.inf, 999)

sku_regimes_fold3['regime']  = sku_regimes_fold3.apply(
    lambda r: classify_regime(r['adi'], r['cv2']), axis=1
)
sku_regimes_fold3['routing'] = sku_regimes_fold3['regime'].map(assign_routing)

# -- Boundary flag: within 10% of either threshold (flag only, no gating applied) -------
sku_regimes_fold3['boundary_flag'] = (
    (sku_regimes_fold3['adi'].between(ADI_THRESHOLD * 0.9, ADI_THRESHOLD * 1.1)) |
    (sku_regimes_fold3['cv2'].between(CV2_THRESHOLD * 0.9, CV2_THRESHOLD * 1.1))
)

print(f'SKUs classified: {len(sku_regimes_fold3):,}')
print()

# -- Fold 3 regime distribution -----------------------------------------------------------
regime_counts_fold3 = sku_regimes_fold3['regime'].value_counts()
print('Fold 3 Regime Distribution:')
print(f'  {"Regime":<15} {"Count":>8} {"% SKUs":>10}')
print(f'  {"-"*35}')
for regime in ['smooth', 'erratic', 'intermittent', 'lumpy']:
    n   = regime_counts_fold3.get(regime, 0)
    pct = n / len(sku_regimes_fold3) * 100
    print(f'  {regime:<15} {n:>8,} {pct:>9.1f}%')
print(f'  {"TOTAL":<15} {len(sku_regimes_fold3):>8,} {"100.0%":>10}')
print()

print(f'Boundary SKUs (within 10% of either threshold): '
      f'{sku_regimes_fold3["boundary_flag"].sum():,} '
      f'({sku_regimes_fold3["boundary_flag"].mean()*100:.1f}%)')
print()

# -- Fold 2 vs Fold 3 distribution comparison (aggregate) --------------------------------
print('Fold 2 vs Fold 3 Regime Counts:')
print(f'  {"Regime":<15} {"Fold 2":>10} {"Fold 3":>10} {"Delta":>10}')
print(f'  {"-"*47}')
for regime in ['smooth', 'erratic', 'intermittent', 'lumpy']:
    f2 = FOLD2_REGIME_COUNTS.get(regime, 0)
    f3 = regime_counts_fold3.get(regime, 0)
    print(f'  {regime:<15} {f2:>10,} {f3:>10,} {f3-f2:>+10,}')
print()

# -- Per-SKU reclassification: requires Fold 2's per-SKU table, not just aggregate counts
fold2_regime_path = f'{SEGMENTATION_DIR}/sku_regimes_fold2.parquet'
assert os.path.exists(fold2_regime_path), f'Missing {fold2_regime_path} -- 06c per-SKU output not found.'
sku_regimes_fold2 = pd.read_parquet(fold2_regime_path)[['id', 'regime']].rename(
    columns={'regime': 'regime_fold2'}
)

regime_compare = sku_regimes_fold3[['id', 'regime', 'boundary_flag']].rename(
    columns={'regime': 'regime_fold3'}
).merge(sku_regimes_fold2, on='id', how='outer', indicator=True)

# Flag SKU-id mismatches explicitly rather than silently dropping them in an outer/inner join
id_mismatch = regime_compare['_merge'] != 'both'
print(f'SKUs present in only one fold\'s table: {id_mismatch.sum():,} '
      f'(should be 0 for a fixed M5 SKU universe -- investigate if nonzero)')
if id_mismatch.sum() > 0:
    print(regime_compare.loc[id_mismatch, ['id', 'regime_fold3', 'regime_fold2', '_merge']].head(10))
print()

regime_compare = regime_compare[regime_compare['_merge'] == 'both'].drop(columns='_merge')
regime_compare['reclassified'] = regime_compare['regime_fold2'] != regime_compare['regime_fold3']

n_reclassified = regime_compare['reclassified'].sum()
print(f'SKUs reclassified between Fold 2 and Fold 3: {n_reclassified:,} '
      f'({n_reclassified/len(regime_compare)*100:.1f}%)')
print()

print('Reclassification transition matrix (Fold 2 regime -> Fold 3 regime, reclassified only):')
transitions = (
    regime_compare[regime_compare['reclassified']]
    .groupby(['regime_fold2', 'regime_fold3'])
    .size()
    .reset_index(name='n_skus')
    .sort_values('n_skus', ascending=False)
)
print(transitions.to_string(index=False))
print()

print(f'Of reclassified SKUs, {regime_compare.loc[regime_compare["reclassified"], "boundary_flag"].mean()*100:.1f}% '
      f'were already boundary_flag=True in Fold 3 (i.e. near a threshold, not a surprise flip).')
print()

# -- Save Fold 3 regime assignments ------------------------------------------------------
regime_fold3_path = f'{SEGMENTATION_DIR}/sku_regimes_fold3.parquet'
sku_regimes_fold3.to_parquet(regime_fold3_path, index=False)
print(f'Saved: {regime_fold3_path}')

Aggregating Fold 3 training data to weekly frequency for ADI/CV² computation...
(Decision frequency = weekly, so classification frequency = weekly)

Weekly training rows: 6,372,410
Weeks per SKU (median): 209

SKUs classified: 30,490

Fold 3 Regime Distribution:
  Regime             Count     % SKUs
  -----------------------------------
  smooth             8,111      26.6%
  erratic              906       3.0%
  intermittent      17,303      56.7%
  lumpy              4,170      13.7%
  TOTAL             30,490     100.0%

Boundary SKUs (within 10% of either threshold): 7,012 (23.0%)

Fold 2 vs Fold 3 Regime Counts:
  Regime              Fold 2     Fold 3      Delta
  -----------------------------------------------
  smooth               8,389      8,111       -278
  erratic                830        906        +76
  intermittent        14,268     17,303     +3,035
  lumpy                7,003      4,170     -2,833

SKUs present in only one fold's table: 0 (should be 0 for a fixed M5 

#### Section 2 Findings — Demand Regime Classification on Fold 3

**Status: Data checked, one methodological caveat flagged below — does not block, but
should be carried into any interpretation of the reclassification numbers.**

- All 30,490 SKUs classified, no dropped or duplicated ids: the Fold2/Fold3 outer-merge
  reported 0 SKUs present in only one table, so the reclassification comparison is on a
  matched, complete universe.
- Fold 3 distribution: smooth 8,111 (26.6%), erratic 906 (3.0%), intermittent 17,303
  (56.7%), lumpy 4,170 (13.7%). Boundary SKUs (within 10% of either threshold): 7,012
  (23.0%) -- notably higher than a "most SKUs sit comfortably inside a quadrant" prior
  would suggest.
- Deltas vs Fold 2 are large and one-directional: intermittent +3,035, lumpy -2,833,
  smooth -278, erratic +76. The transition matrix is internally consistent with these
  deltas (lumpy outflow of 3,702 SKUs minus inflow of 869 = -2,833; intermittent inflow
  of 4,257 minus outflow of 1,222 = +3,035), so this isn't a bookkeeping bug -- it's a
  real, large shift in measured regime.
- 19.6% of SKUs (5,987) reclassified between folds. The single largest transition by far
  is lumpy -> intermittent (3,651 SKUs, 61% of all reclassifications) -- essentially one
  dominant migration pattern, not diffuse noise across all four regimes.
- Only 36.1% of reclassified SKUs were boundary_flag=True in Fold 3. That means nearly
  two-thirds of reclassified SKUs moved decisively, not as a borderline flicker -- which
  is itself evidence that a hysteresis gate would not have suppressed most of this
  movement anyway, since hysteresis is designed to dampen noise near a threshold, not
  large, non-boundary jumps. This weakly reinforces yesterday's boundary_flag-only
  decision: hysteresis wasn't the missing piece for reproducing Fold 2 stability here.

**Caveat to flag, not yet resolved:** Fold 2's ADI/CV² were computed over a ~3-year
training window (2011-02-02 -> 2014-01-31, per 06c), while Fold 3's are computed over a
~4-year window (2011-02-02 -> 2015-01-31, per Section 1) -- one additional year of
history feeds into every SKU's weekly aggregation. Since ADI is literally
`total_weeks / nonzero_weeks`, extending the lookback window mechanically raises ADI for
any SKU with sparse or late-starting sales early in its history, independent of any real
change in recent demand behavior. The dominant lumpy -> intermittent migration (both are
adi >= 1.32 regimes, differing only on cv2) is at least directionally consistent with an
artifact of a longer window rather than a genuine behavior shift, but this is a
plausible explanation, not a confirmed one -- I haven't (and given the "run once" rule,
can't cheaply) re-run Fold 3's classification on a matched 3-year window to isolate window
length from real drift. Recommend carrying this caveat forward into any downstream
section that treats the Fold 2 vs Fold 3 regime comparison as evidence of routing
instability, rather than presenting the 19.6% reclassification rate as a clean stability
metric on its own.

## Section 3 — Retrain Production Model on Fold 3 Training Data

Retrains the frozen Tweedie architecture (BEST_PARAMS_TWEEDIE, loaded from
`tweedie_best_params.pkl`, not redefined here) on the Fold 3 training window.
No Optuna, no hyperparameter changes -- 06d adopted none of its three
experiments (Section 5 above), so this is the original 06/06b baseline
architecture. Train/monitor split reused verbatim from 06_lightgbm_demand.ipynb's
`FOLDS['fold_3']` (monitor_start=2014-12-02 through train_end=2015-01-31),
matching the same monitor-holdout pattern used to train every prior fold's
model. `fold3_val` is not touched in this section -- reserved for Section 5.

In [5]:
# -- 07 Section 3: Retrain Production Model on Fold 3 Training Data ---------------------
import time

# -- Frozen hyperparameters -- loaded from disk, not redefined, per standing rule -------
with open(f'{CALIBRATION_DIR}/tweedie_best_params.pkl', 'rb') as f:
    BEST_PARAMS_TWEEDIE = pickle.load(f)

print('Loaded frozen Tweedie hyperparameters (tweedie_best_params.pkl):')
for k, v in BEST_PARAMS_TWEEDIE.items():
    print(f'  {k:<26} {v}')
print()

# -- Feature columns -- same artifact used by every prior notebook ---------------------
with open(f'{FEATURES_DIR}/feature_cols_v2.pkl', 'rb') as f:
    FEATURE_COLS = pickle.load(f)
print(f'Feature columns: {len(FEATURE_COLS)}')
print()

TARGET_COL = 'target'

# -- Fold 3 train/monitor split -- reused verbatim from 06_lightgbm_demand.ipynb's
# FOLDS['fold_3']. fold3_val is intentionally NOT sliced here -- Section 5 only. --------
FOLD3_TRAIN_START   = '2011-02-01'
FOLD3_MONITOR_START = '2014-12-02'
FOLD3_TRAIN_END     = '2015-01-31'

train_sub   = fold3_train[(fold3_train['date'] >= FOLD3_TRAIN_START) &
                           (fold3_train['date'] <  FOLD3_MONITOR_START)].copy()
monitor_sub = fold3_train[(fold3_train['date'] >= FOLD3_MONITOR_START) &
                           (fold3_train['date'] <= FOLD3_TRAIN_END)].copy()

print(f'Train sub-window   : {train_sub["date"].min().date()} -> {train_sub["date"].max().date()} '
      f'({len(train_sub):,} rows)')
print(f'Monitor sub-window : {monitor_sub["date"].min().date()} -> {monitor_sub["date"].max().date()} '
      f'({len(monitor_sub):,} rows)')
print()

# -- Leakage assertions -- monitor must not spill into fold3_val, train must not spill
# into monitor. Mirrors the assertion style already used in Section 1. -----------------
assert train_sub['date'].max() < pd.Timestamp(FOLD3_MONITOR_START), 'Train/monitor overlap'
assert monitor_sub['date'].max() < pd.Timestamp('2015-02-01'), 'Monitor spills into Fold 3 val'
print('Train/monitor split assertions passed.')
print()

# -- Build targets -- Tweedie trains in raw unit space via expm1(log target), same as
# every prior fold (06_lightgbm_demand.ipynb cell 16 / cell 21). -----------------------
X_train_tw = train_sub[FEATURE_COLS]
y_train_tw = np.expm1(train_sub[TARGET_COL])
X_monitor_tw = monitor_sub[FEATURE_COLS]
y_monitor_tw = np.expm1(monitor_sub[TARGET_COL])

print('Building Fold 3 Tweedie datasets...')
dtrain_fold3 = lgb.Dataset(X_train_tw, label=y_train_tw, feature_name=list(FEATURE_COLS),
                            free_raw_data=False)
dtrain_fold3.construct()
dmonitor_fold3 = lgb.Dataset(X_monitor_tw, label=y_monitor_tw, reference=dtrain_fold3,
                              free_raw_data=False)
dmonitor_fold3.construct()
print('Datasets ready.')
print()

EARLY_STOPPING_ROUNDS = 50  # locked in 06_lightgbm_demand.ipynb, reused verbatim

print('Training Fold 3 Tweedie model with frozen BEST_PARAMS_TWEEDIE...')
t0 = time.time()
model_tweedie_fold3 = lgb.train(
    params=BEST_PARAMS_TWEEDIE,
    train_set=dtrain_fold3,
    num_boost_round=2000,
    valid_sets=[dmonitor_fold3],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)
elapsed = time.time() - t0

print(f'\nDone in {elapsed:.1f}s  |  best n_estimators: {model_tweedie_fold3.best_iteration}')
print()

# -- Monitor-set diagnostic ONLY -- fold3_val is never predicted on in this section -----
y_pred_monitor = model_tweedie_fold3.predict(X_monitor_tw)
mask_monitor   = y_monitor_tw.values > 0
rmse_monitor   = np.sqrt(mean_squared_error(y_monitor_tw.values[mask_monitor],
                                             y_pred_monitor[mask_monitor]))
print(f'Monitor-set unit-RMSE (diagnostic only, not a val metric): {rmse_monitor:.2f}')
print()

# -- Save model ---------------------------------------------------------------------
model_path_fold3 = f'{MODELS_DIR}/tweedie_optimized_fold3.txt'
model_tweedie_fold3.save_model(model_path_fold3)
print(f'Saved: {model_path_fold3}')

Loaded frozen Tweedie hyperparameters (tweedie_best_params.pkl):
  num_leaves                 224
  learning_rate              0.017
  feature_fraction           0.86
  bagging_fraction           0.86
  bagging_freq               2
  min_child_samples          50
  reg_alpha                  4.78
  reg_lambda                 1.94
  tweedie_variance_power     1.02
  objective                  tweedie
  metric                     tweedie
  verbosity                  -1
  random_state               42
  num_threads                -1

Feature columns: 39

Train sub-window   : 2011-02-02 -> 2014-12-01 (27,140,570 rows)
Monitor sub-window : 2014-12-02 -> 2015-01-31 (1,559,244 rows)

Train/monitor split assertions passed.

Building Fold 3 Tweedie datasets...
Datasets ready.

Training Fold 3 Tweedie model with frozen BEST_PARAMS_TWEEDIE...
[100]	valid_0's tweedie: 65.6928
[200]	valid_0's tweedie: 65.6278
[300]	valid_0's tweedie: 65.6175
[400]	valid_0's tweedie: 65.6133
[500]	valid_0's tweedie:

#### Section 3 Findings — Retrain Production Model on Fold 3 Training Data

**Status: LOCKED. No rerun needed.**

**Split integrity — exact match, not approximate:** `train_sub` (27,140,570 rows,
2011-02-02→2014-12-01) and `monitor_sub` (1,559,244 rows, 2014-12-02→2015-01-31) match
06's `fold_3` split definition to the row, confirming this reuses the canonical fold
boundaries rather than an ad hoc recomputation. Leakage assertions passed: train/monitor
boundary clean, monitor ends before Fold 3 val's 2015-02-01 start. Fold 3 val itself was
not touched — correct, that's Section 5's job, once.

**Frozen hyperparameters confirmed unchanged from `tweedie_best_params.pkl`** — same
`num_leaves=224`, `lr=0.017`, `tweedie_variance_power=1.02`, etc. as 06b. No retuning
occurred, as required.

**Convergence — one thing to note, not a problem:** training hit the `num_boost_round=2000`
cap without early stopping triggering (`best_iteration=2000`), vs. Fold 2's model which
early-stopped at 1440. The tweedie metric was still improving at round 2000, but the
improvement had flattened hard (65.6928 → 65.6052 across all 2000 rounds, a ~0.13% move,
with round-to-round deltas below 0.0002 by round 1000). Functionally converged; the cap
just arrived before the 50-round-patience trigger did. Since hyperparameters are frozen by
design, this isn't something to fix — flagging only so it's not mistaken for early stopping
having failed.

**On "is it better than Fold 2" — not a valid comparison as computed, and that's fine:**
the 2.91 monitor-set unit-RMSE is diagnostic-only, computed on a 2-month internal carve-out
from *training* data, not against a held-out val year. Fold 2's comparable numbers (3.10
raw / 3.0262 calibrated) came from the actual, full Fold 2 val set. Different population,
different window length, different purpose — a lower number here says the model fits its
own monitor slice reasonably, not that Fold 3 will outperform Fold 2. The real comparison
happens in Section 6, against Fold 3 val, once Section 5 runs.

**Training time:** 1467s vs. Fold 2's 721s — roughly 2× on ~1.5× the row count, consistent
with tree depth/complexity at `num_leaves=224`, not a red flag.

**Bottom line:** split is leak-free and verified against the canonical fold definition,
hyperparameters are correctly frozen, model trained to convergence. `tweedie_optimized_fold3.txt`
saved and ready to feed Section 4. Proceed — nothing here warrants a rerun.

## Section 4 — Conformal Calibration on Fold 3

Refit per-SKU conformal residual distributions on Fold 3, using the method locked in
`06e_uncertainty_quantification.ipynb` (signed residuals, per-SKU empirical quantiles,
`MIN_RESIDUALS_PER_SKU=5` floor). What's re-derived here is the residual distribution only
-- the service-level choice per regime (`SERVICE_LEVEL_BY_REGIME` / `CONFORMAL_ALPHA`,
loaded in Section 1) stays locked from 06e's economic + empirical-elbow analysis on Fold 2.
Re-running that selection layer against Fold 3 would be re-tuning against the same kind of
evidence 06e already settled, not recomputing a fold-specific quantity.

**Residual source, and why it's `monitor_sub` and not `train_sub`:** `train_sub` fit
`model_tweedie_fold3`, so residuals computed on it would be in-sample and artificially
tight -- narrower intervals than the model will actually produce on unseen data, which is
the wrong direction to be wrong in for a safety-stock input. `monitor_sub` is genuinely
unseen by the model (early-stopping uses it to pick `best_iteration`, not to fit weights)
and was never touched again after Section 3, so it's the correct population.

**Deviation from 06e's own structure, flagged explicitly:** 06e had a full Fold 2 val year
to spend, so it split it chronologically into a calibration window and a held-out
coverage-test window, then checked empirical coverage against target. Fold 3 has no
equivalent slack -- `fold3_val` is off-limits until Section 5, and `monitor_sub` is only
~60 days. Splitting that further would push per-SKU residual counts below the floor for a
meaningful share of SKUs, for a coverage check on an already-small sample. Instead: use all
of `monitor_sub` to build the residual distributions (maximize sample size), and replace
the statistical coverage-test gate with a structural sanity check (pooled shape, degenerate
SKUs, monotonic width growth). The real coverage evidence comes in Section 6, against
`fold3_val`, as a byproduct of Section 5's one-shot run -- not duplicated here.


In [7]:
# -- 07 Section 4: Conformal Calibration on Fold 3 ---------------------------------------
# Method identical to 06e (signed per-SKU residuals, empirical quantiles). NOT re-derived:
# SERVICE_LEVEL_BY_REGIME / CONFORMAL_ALPHA (locked, Section 1, from 06e's Fold 2 analysis).
# See markdown above for why this uses monitor_sub (not train_sub or fold3_val) and why the
# coverage-test split from 06e is replaced with a sanity check here.

MIN_RESIDUALS_PER_SKU = 5  # locked, 06e

# -- Restrict to Fold 3's tweedie-routed SKUs (smooth + erratic). Reuses model outputs
# already computed in Section 3 -- no re-prediction against monitor_sub. ----------------
tweedie_skus_fold3 = set(sku_regimes_fold3.loc[sku_regimes_fold3['routing'] == 'tweedie', 'id'])
regime_lookup_fold3 = sku_regimes_fold3.set_index('id')['regime']

monitor_id       = monitor_sub['id'].reset_index(drop=True)
yhat_monitor     = np.maximum(y_pred_monitor, 0)              # clip at 0, mirrors 06e cell 1
target_monitor   = y_monitor_tw.reset_index(drop=True).values
residual_monitor = target_monitor - yhat_monitor

resid_df = pd.DataFrame({
    'id':       monitor_id.values,
    'target':   target_monitor,
    'yhat':     yhat_monitor,
    'residual': residual_monitor,
})
resid_df = resid_df[resid_df['id'].isin(tweedie_skus_fold3)]

print(f'Monitor rows, tweedie-routed SKUs only: {len(resid_df):,} / {len(monitor_sub):,}')
print(f'Tweedie-routed SKUs (Fold 3, smooth+erratic): {len(tweedie_skus_fold3):,}')
print()

# -- Per-SKU residual distributions -- ALL of monitor_sub, no calib/test split (see
# markdown above for why). ---------------------------------------------------------------
sku_residuals_fold3 = resid_df.groupby('id', observed=True)['residual'].apply(list).to_dict()

n_skus_with_data   = len(sku_residuals_fold3)
n_skus_below_floor = sum(1 for v in sku_residuals_fold3.values() if len(v) < MIN_RESIDUALS_PER_SKU)
n_skus_missing     = len(tweedie_skus_fold3) - n_skus_with_data

print(f'Tweedie SKUs with residual data: {n_skus_with_data:,} / {len(tweedie_skus_fold3):,}')
print(f'  Missing entirely from monitor_sub: {n_skus_missing:,}')
print(f'  Below MIN_RESIDUALS_PER_SKU={MIN_RESIDUALS_PER_SKU} floor (will be skipped downstream): {n_skus_below_floor:,}')
print()

# -- Sanity checks -- replaces 06e's statistical coverage-test gate; see markdown -------
pooled = pd.Series([r for v in sku_residuals_fold3.values() for r in v])
print(f'Pooled residual distribution (n={len(pooled):,}):')
print(pooled.describe())
print(f'skew: {pooled.skew():.3f}')
print()

valid_skus = {k: v for k, v in sku_residuals_fold3.items() if len(v) >= MIN_RESIDUALS_PER_SKU}
width_check = pd.DataFrame({
    sku: {f'q{int(l*100)}': np.percentile(v, l * 100) for l in [0.50, 0.75, 0.90, 0.99]}
    for sku, v in valid_skus.items()
}).T

n_inverted = ((width_check['q90'] < width_check['q50']) |
              (width_check['q99'] < width_check['q90'])).sum()
n_negative_q90 = (width_check['q90'] < 0).sum()

print(f'SKUs with non-monotonic quantile widths (q90<q50 or q99<q90): {n_inverted:,} / {len(width_check):,}')
print(f'SKUs with negative q90 width: {n_negative_q90:,}')
print()

# -- Regime-level width summary, for context only -- NOT used to pick service levels ----
width_by_regime = width_check.join(regime_lookup_fold3).groupby('regime')[
    ['q50', 'q75', 'q90', 'q99']
].median()
print('Median interval width by regime (context only, service levels stay locked from 06e):')
print(width_by_regime.round(2))
print()

# -- Save Fold 3 conformal residuals -- same locked service levels/alpha as 06e, only the
# residual distributions themselves are new. --------------------------------------------
conformal_fold3 = {
    'sku_residuals':         sku_residuals_fold3,
    'default_service_level': SERVICE_LEVEL_BY_REGIME,   # locked from 06e, not re-derived
    'residual_unit':         conformal_fold2['residual_unit'],
    'winner_model':          PRODUCTION_MODEL,
    'min_residuals_per_sku': MIN_RESIDUALS_PER_SKU,
    'source_window':         f'{FOLD3_MONITOR_START} -> {FOLD3_TRAIN_END} (monitor_sub, in-sample)',
}
conformal_fold3_path = f'{CALIBRATION_DIR}/conformal_residuals_fold3.pkl'
with open(conformal_fold3_path, 'wb') as f:
    pickle.dump(conformal_fold3, f)
print(f'Saved: {conformal_fold3_path}')


Monitor rows, tweedie-routed SKUs only: 521,340 / 1,559,244
Tweedie-routed SKUs (Fold 3, smooth+erratic): 9,017

Tweedie SKUs with residual data: 8,895 / 9,017
  Missing entirely from monitor_sub: 122
  Below MIN_RESIDUALS_PER_SKU=5 floor (will be skipped downstream): 32

Pooled residual distribution (n=521,340):
count    521340.000000
mean          0.000110
std           2.665811
min         -79.764997
25%          -0.760709
50%          -0.279313
75%           0.539849
max         299.453947
dtype: float64
skew: 10.621

SKUs with non-monotonic quantile widths (q90<q50 or q99<q90): 0 / 8,863
SKUs with negative q90 width: 294

Median interval width by regime (context only, service levels stay locked from 06e):
          q50   q75   q90   q99
regime                         
erratic -0.35  0.40  1.38  3.59
smooth  -0.29  0.49  1.40  3.28

Saved: ../data/processed/calibration/conformal_residuals_fold3.pkl


#### Section 4 Findings — Conformal Calibration on Fold 3

**Status: LOCKED.**

**Bug found & fixed:** `id` is a categorical column carried through from the parquet
schema. `resid_df.groupby('id')['residual'].apply(list)` without `observed=True`
enumerated every category in the full 30,490-SKU codebook, not just the 9,017 tweedie-
routed SKUs actually present after filtering — producing a `30,490 / 9,017` "with data"
count and a structurally negative "missing" count. Same class of bug as 06g Section 1.
Fixed with `groupby('id', observed=True)`.

**Coverage, post-fix:** 8,895 / 9,017 tweedie SKUs (98.6%) have residual data in
`monitor_sub`; 122 are missing entirely (never appeared in the ~60-day monitor window —
plausible for very low-frequency Erratic SKUs, not investigated further since 06g's
pooled-by-regime fallback covers them). Of the 8,895, 32 fall below
`MIN_RESIDUALS_PER_SKU=5` and will use the regime-pooled fallback quantile in 06g's
Section 1 logic rather than their own distribution — a small, expected fraction (0.4% of
the routed population), not a red flag.

**Sanity checks pass:** 0/8,863 valid SKUs show non-monotonic quantile widths (q90<q50 or
q99<q90) — empirical quantiles are well-behaved. 294 SKUs have negative q90 width, meaning
even the 90th-percentile residual undershoots the forecast; this mirrors 06e's own finding
and is handled downstream by `resid_q = max(resid_q, 0.0)` in 06g's reorder-point calc —
not a defect here.

**Pooled residual distribution** (n=521,340): median −0.28, right-skewed (skew 10.6, max
299.5) — consistent with Tweedie's typical-day-overshoot / occasional-spike pattern seen
in every prior fold. No structural change vs. Fold 2's shape.

**What's explicitly not re-derived:** `SERVICE_LEVEL_BY_REGIME` / `CONFORMAL_ALPHA` stay
locked from 06e's Fold 2 analysis, per the plan. Section 4 only refit the residual
distributions themselves.

Saved: `conformal_residuals_fold3.pkl`. No open items — Section 4 is a valid input to
Section 5.

## Section 5 — Run Full Pipeline (Fold 3)

**Scope:** build the Fold 3 equivalent of every artifact 06f/06g produced for Fold 2, then run
all three inventory policies (naive, static (s,Q), dynamic periodic-review) over Fold 3's actual
validation window. Sections 1-4 above only covered Smooth/Erratic (Tweedie+conformal); this
section adds Lumpy (historical min-max policy) and Intermittent (TSB + block-bootstrap), merges
all four regimes, and runs the static and dynamic simulations.

**Reused verbatim, not re-derived:**
- `ALPHA_LOCKED`/`BETA_LOCKED` — 06f Section 3's winning TSB params (`tsb_winning_params.pkl`).
- `get_buffer_multiplier` / Lumpy's min-max formula — 06f Section 4, unchanged.
- `block_bootstrap_lead_time_demand` — 06f Section 6's moving-block bootstrap mechanism.
- `simulate_inventory`, `simulate_inventory_periodic`, `dynamic_safety_stock`,
  `production_trajectory`, `tsb_walkforward_weekly` — 06g Sections 2 and 6/7, unchanged logic.
- Cost model (2.0x stockout / 25% carrying default, real per-SKU `sell_price`) — 06g Section 4/8.
- `SERVICE_LEVEL_BY_REGIME` / `CONFORMAL_ALPHA` — locked in Section 1 from 06e, not re-derived.

**Deliberate deviation from 06f Section 6:** 06f's "Stage 2: production numbers — refit on ALL
available history" used the *entire* raw sales file for Fold 2's Lumpy/Intermittent policy fit,
which silently extended past Fold 2's own val window (harmless there, since Fold 2 wasn't the
final gate). Fold 3 **is** the final gate — the same convention here would let Fold 3's own val
window leak into the policy that gets evaluated against it. Both Lumpy's weekly aggregation and
Intermittent's TSB/bootstrap history are bounded strictly to `fold3_train` (through
`FOLD3_TRAIN_END = 2015-01-31`) instead.

**Not re-validated for Fold 3 (open items, inherited from Fold 2's own caveats):**
- Smooth/Erratic's `sqrt(7)`-scaling of daily forecast-error residuals to a 7-day lead time
  (06g's Pre-flight #4 rolling-origin coverage check was never rerun on Fold 3 data here).
- Intermittent's block-bootstrap coverage (06f Section 6 Stage 1's rolling-origin validation
  was a one-time method check on Fold 2; not rerun here — the *method* is locked, not re-tested).

Neither invalidates Section 5's output; both are correctness caveats to carry into Section 6's
write-up, same as 06g carried its own Pre-flight #4 flag through to Section 7.

**Data dependency:** simulation needs `sales_train_validation.csv` to actually cover Fold 3's val
window (through ~2016-01-31), not just Fold 2's (through 2015-01-31). Checked explicitly below
rather than assumed — if it fails, swap in `sales_train_evaluation.csv`.

In [23]:
# -- 07 Section 5: Run Full Pipeline (Fold 3) --------------------------------------------
import time

RAW_DIR = '../data/raw'
LEAD_TIME_DAYS = 7
LEAD_TIME_WEEKS = 1
REVIEW_PERIOD_WEEKS = 1
STOCKOUT_MULT = 2.0
CARRY_RATE = 0.25
STOCKOUT_MULTIPLIERS = [1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
CARRYING_RATES = [0.15, 0.20, 0.25, 0.30]
SL_ORDER = ['q50', 'q75', 'q80', 'q90', 'q95', 'q99']
SERVICE_LEVELS_SWEEP = [0.50, 0.75, 0.80, 0.90, 0.95, 0.99]


def service_level_frac(regime):
    label = SERVICE_LEVEL_BY_REGIME.get(regime, 'q80')
    return int(label[1:]) / 100


FOLD3_TRAIN_END_TS = pd.Timestamp(FOLD3_TRAIN_END)   # locked, Section 3

# ══════════════════════════════════════════════════════════════════════════════════════
# 5.1 — Smooth/Erratic reorder params, Fold 3 (mirrors 06g Section 1)
# Reuses Section 4's resid_df/sku_residuals_fold3 directly -- no re-prediction needed,
# monitor_sub's yhat/residuals ARE the calibration source for Fold 3 (per Section 4 markdown).
# ══════════════════════════════════════════════════════════════════════════════════════

point_forecast_fold3 = resid_df.groupby('id', observed=True)['yhat'].mean()

pooled_residuals_by_regime_fold3 = {}
for regime in ('smooth', 'erratic'):
    ids_in_regime = {
        sid for sid, r in regime_lookup_fold3.items()
        if r == regime
    }
    pooled = np.concatenate([
        np.asarray(v)
        for sid, v in sku_residuals_fold3.items()
        if sid in ids_in_regime
    ]) if ids_in_regime else np.array([0.0])
    pooled_residuals_by_regime_fold3[regime] = pooled

# Fold 3's PRODUCTION_MODEL ('tweedie_baseline') predicts next-day units, not a 7-day-forward
# sum -- confirmed in Section 1 (not in 06g's SEVEN_DAY_TARGET_EXPERIMENTS set). sqrt(7) scaling
# applies, same UNVALIDATED-for-this-fold caveat as 06g's Pre-flight #4.
TARGET_IS_7DAY_FOLD3 = False

rows = []
n_negative_q, n_nan_forecast, n_zero_history_excluded = 0, 0, 0
nan_forecast_ids = []

for sku_id in sorted(tweedie_skus_fold3):
    if sku_id not in point_forecast_fold3.index:
        n_zero_history_excluded += 1
        continue

    regime = regime_lookup_fold3[sku_id]
    sl = service_level_frac(regime)
    pf_native = point_forecast_fold3[sku_id]
    residuals = sku_residuals_fold3.get(sku_id, [])

    low_conf = len(residuals) < MIN_RESIDUALS_PER_SKU

    if low_conf:
        resid_q = np.percentile(
            pooled_residuals_by_regime_fold3.get(regime, [0.0]),
            sl * 100
        )
    else:
        resid_q = np.percentile(residuals, sl * 100)

    if resid_q < 0:
        n_negative_q += 1

    resid_q = max(resid_q, 0.0)

    expected_lead_time_demand = pf_native * LEAD_TIME_DAYS
    safety_stock = resid_q * np.sqrt(LEAD_TIME_DAYS)

    if not (
        np.isfinite(expected_lead_time_demand)
        and np.isfinite(safety_stock)
    ):
        n_nan_forecast += 1
        nan_forecast_ids.append(sku_id)
        low_conf = True
        fallback_required = True

        expected_lead_time_demand = 0.0
        safety_stock = 0.0
        reorder_point = 0.0
        order_qty = 1.0
    else:
        fallback_required = low_conf
        reorder_point = expected_lead_time_demand + safety_stock
        order_qty = max(expected_lead_time_demand, 1.0)

    rows.append({
        'id': sku_id,
        'regime': regime,
        'method': 'conformal_tweedie',
        'expected_lead_time_demand': expected_lead_time_demand,
        'reorder_point': reorder_point,
        'safety_buffer': safety_stock,
        'order_qty': order_qty,
        'low_confidence': low_conf,
        'fallback_required': fallback_required,
        'sqrt_scaling_unvalidated': not TARGET_IS_7DAY_FOLD3,
        'service_level': sl,
    })

smooth_erratic_unified_fold3 = pd.DataFrame(rows)

assert smooth_erratic_unified_fold3['order_qty'].notna().all() and \
    (smooth_erratic_unified_fold3['order_qty'] > 0).all(), \
    'BUG: some Smooth/Erratic Fold 3 SKU has order_qty <= 0.'

print(
    f'Smooth/Erratic Fold 3 reorder table: '
    f'{len(smooth_erratic_unified_fold3):,} SKUs '
    f'({n_zero_history_excluded:,} excluded, no monitor-window data; '
    f'{n_nan_forecast:,} NaN-guarded)'
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.1b — Dense raw M5 demand history for inventory calibration
#
# fold3_train is an engineered feature panel and does not contain every (id, date)
# demand observation. Section 5's Lumpy/Intermittent demand histories therefore use
# the dense raw M5 sales panel for the exact Fold 3 training dates.
# ══════════════════════════════════════════════════════════════════════════════════════

calendar_df = pd.read_csv(f'{RAW_DIR}/calendar.csv')
calendar_df['date'] = pd.to_datetime(calendar_df['date'])
day_to_date = calendar_df.set_index('d')['date'].to_dict()

raw_sales = pd.read_csv(f'{RAW_DIR}/sales_train_validation.csv')
day_cols_all = [c for c in raw_sales.columns if c.startswith('d_')]

fold3_history_dates = pd.DatetimeIndex(
    sorted(fold3_train['date'].unique())
)
fold3_history_date_set = set(fold3_history_dates)

fold3_history_day_cols = [
    d
    for d in day_cols_all
    if d in day_to_date and day_to_date[d] in fold3_history_date_set
]

fold3_history_dates_from_raw = pd.DatetimeIndex([
    day_to_date[d]
    for d in fold3_history_day_cols
])

assert len(fold3_history_day_cols) == len(fold3_history_dates), (
    f'Raw M5 Fold 3 history mismatch: '
    f'{len(fold3_history_day_cols)} raw days vs '
    f'{len(fold3_history_dates)} Fold 3 dates.'
)

assert set(fold3_history_dates_from_raw) == set(fold3_history_dates), (
    'Raw M5 history dates do not exactly match Fold 3 training dates.'
)

raw_sales_indexed_fold3 = raw_sales.set_index('id')

print(
    f'\nDense raw Fold 3 demand history: '
    f'{len(fold3_history_dates):,} days '
    f'({fold3_history_dates.min().date()} -> '
    f'{fold3_history_dates.max().date()})'
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.2 — Lumpy policy, Fold 3
#
# IMPORTANT:
# Uses dense raw M5 demand rather than fold3_train because the feature panel drops
# observed (id, date) demand rows. Using fold3_train here would silently undercount
# positive demand.
# ══════════════════════════════════════════════════════════════════════════════════════

lumpy_ids_fold3 = sku_regimes_fold3.loc[
    sku_regimes_fold3['regime'] == 'lumpy',
    'id'
].tolist()

print(f'\nLumpy SKUs (Fold 3): {len(lumpy_ids_fold3):,}')


def get_buffer_multiplier(max_weekly):
    if max_weekly == 0:
        return 2.0
    elif max_weekly <= 5:
        return 1.5
    else:
        return 1.25


def historical_policy_forecast_from_weekly(
    weekly_demand,
    lead_time_weeks=1
):
    max_weekly = weekly_demand.max()
    avg_weekly = weekly_demand.mean()
    buffer = get_buffer_multiplier(max_weekly)

    reorder_point = max_weekly * lead_time_weeks * buffer

    MIN_ORDER_ZERO, MIN_ORDER_ACTIVE = 5, 1

    if max_weekly == 0:
        order_qty = MIN_ORDER_ZERO
    else:
        order_qty = max(
            max_weekly * buffer - avg_weekly,
            avg_weekly,
            MIN_ORDER_ACTIVE
        )

    return {
        'reorder_point': reorder_point,
        'order_qty': order_qty,
        'max_weekly': max_weekly,
        'avg_weekly': avg_weekly,
        'buffer_multiplier': buffer,
    }


# Dense raw daily history for Lumpy SKUs.
lumpy_raw_fold3 = (
    raw_sales_indexed_fold3
    .loc[lumpy_ids_fold3, fold3_history_day_cols]
    .copy()
)

lumpy_raw_fold3.columns = fold3_history_dates_from_raw

assert not lumpy_raw_fold3.isna().any().any(), (
    'BUG: raw M5 Lumpy history contains NaNs.'
)

# Daily -> weekly demand.
lumpy_weekly_matrix_fold3 = (
    lumpy_raw_fold3
    .T
    .resample('W')
    .sum()
    .T
)

lumpy_policy_params_fold3 = {
    sku_id: historical_policy_forecast_from_weekly(
        lumpy_weekly_matrix_fold3.loc[sku_id],
        lead_time_weeks=1
    )
    for sku_id in lumpy_ids_fold3
}

lumpy_policy_df_fold3 = pd.DataFrame([
    {'id': sid, **p}
    for sid, p in lumpy_policy_params_fold3.items()
])

lumpy_policy_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/lumpy_policy_params_fold3.parquet'
)

n_zero_lumpy = (
    lumpy_policy_df_fold3['max_weekly'] == 0
).sum()

print(
    f'Lumpy policy computed for '
    f'{len(lumpy_policy_df_fold3):,} SKUs '
    f'({n_zero_lumpy:,} zero-demand-in-training, '
    f'floored at MIN_ORDER_ZERO=5)'
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.3 — Intermittent: TSB + block-bootstrap safety stock, Fold 3
#
# ALPHA_LOCKED/BETA_LOCKED reused verbatim from 06f.
# IMPORTANT:
# Uses dense raw M5 daily history rather than pivoting fold3_train.
# ══════════════════════════════════════════════════════════════════════════════════════

intermittent_ids_fold3 = sku_regimes_fold3.loc[
    sku_regimes_fold3['regime'] == 'intermittent',
    'id'
].tolist()

print(
    f'\nIntermittent SKUs (Fold 3): '
    f'{len(intermittent_ids_fold3):,}'
)

with open(
    f'{MODELS_DIR}/tsb_winning_params.pkl',
    'rb'
) as f:
    TSB_PARAMS = pickle.load(f)

ALPHA_LOCKED = TSB_PARAMS['alpha']
BETA_LOCKED = TSB_PARAMS['beta']

print(
    f'Loaded locked TSB params (06f, not re-derived): '
    f'alpha={ALPHA_LOCKED}, beta={BETA_LOCKED}'
)


def tsb_forecast(demand, alpha, beta):
    n = len(demand)

    p = np.zeros(n)
    z = np.zeros(n)
    forecast = np.zeros(n)

    nonzero_mask = demand > 0

    if not nonzero_mask.any():
        return forecast, 0.0, 0.0

    first_nonzero_idx = np.argmax(nonzero_mask)

    z[0] = demand[first_nonzero_idx]
    p[0] = nonzero_mask.mean()
    forecast[0] = p[0] * z[0]

    for t in range(1, n):
        occurred = demand[t - 1] > 0

        p[t] = (
            beta * occurred
            + (1 - beta) * p[t - 1]
        )

        z[t] = (
            alpha * demand[t - 1]
            + (1 - alpha) * z[t - 1]
            if occurred
            else z[t - 1]
        )

        forecast[t] = p[t] * z[t]

    return forecast, p[-1], z[-1]


def block_bootstrap_lead_time_demand(
    demand_history_causal,
    lead_time_days,
    n_sims,
    rng,
    recency_days=180
):
    pool = (
        demand_history_causal[-recency_days:]
        if len(demand_history_causal) > recency_days
        else demand_history_causal
    )

    n = len(pool)

    if n < lead_time_days:
        return None

    n_blocks = n - lead_time_days + 1

    starts = rng.integers(
        0,
        n_blocks,
        size=n_sims
    )

    cumsum = np.concatenate([
        [0],
        np.cumsum(pool)
    ])

    block_sums = (
        cumsum[starts + lead_time_days]
        - cumsum[starts]
    )

    return block_sums, n_blocks


# Dense raw M5 demand matrix.
intermittent_wide_fold3 = (
    raw_sales_indexed_fold3
    .loc[intermittent_ids_fold3, fold3_history_day_cols]
    .copy()
)

intermittent_wide_fold3.columns = fold3_history_dates_from_raw
intermittent_wide_fold3 = intermittent_wide_fold3.sort_index(axis=1)

assert intermittent_wide_fold3.shape == (
    len(intermittent_ids_fold3),
    len(fold3_history_dates)
), (
    f'Unexpected intermittent demand matrix shape: '
    f'{intermittent_wide_fold3.shape}'
)

assert not intermittent_wide_fold3.isna().any().any(), (
    'BUG: raw M5 intermittent demand history contains NaNs.'
)

intermittent_ids_ordered_fold3 = (
    intermittent_wide_fold3.index.tolist()
)

demand_matrix_fold3 = (
    intermittent_wide_fold3.to_numpy(dtype=float)
)

print(
    f'Dense intermittent demand matrix: '
    f'{demand_matrix_fold3.shape[0]:,} SKUs x '
    f'{demand_matrix_fold3.shape[1]:,} days'
)

N_BOOTSTRAP = 1000
BLOCK_RECENCY_DAYS = 180
MIN_BLOCKS_FOR_CONFIDENCE = 10

MIN_ORDER_ZERO_INTERMITTENT = 5
MIN_ORDER_ACTIVE_INTERMITTENT = 1

rng_boot = np.random.default_rng(42)

safety_stock_records_fold3 = []

for i, sku_id in enumerate(intermittent_ids_ordered_fold3):

    full_history = demand_matrix_fold3[i]

    _, p_final, z_final = tsb_forecast(
        full_history,
        ALPHA_LOCKED,
        BETA_LOCKED
    )

    result = block_bootstrap_lead_time_demand(
        full_history,
        LEAD_TIME_DAYS,
        N_BOOTSTRAP,
        rng_boot,
        BLOCK_RECENCY_DAYS
    )

    if result is None:
        safety_stock_records_fold3.append({
            'id': sku_id,
            'mean_lead_time_demand': 0.0,
            'safety_stock_q80': 0.0,
            'p_final': p_final,
            'z_final': z_final,
            'order_qty': MIN_ORDER_ZERO_INTERMITTENT,
            'n_blocks_available': 0,
            'low_confidence': True,
            'fallback_required': True,
        })
        continue

    sim, n_blocks = result

    order_qty = max(
        z_final,
        MIN_ORDER_ACTIVE_INTERMITTENT
    )

    record = {
        'id': sku_id,
        'mean_lead_time_demand': float(sim.mean()),
        'p_final': p_final,
        'z_final': z_final,
        'order_qty': float(order_qty),
        'n_blocks_available': n_blocks,
        'low_confidence': n_blocks < MIN_BLOCKS_FOR_CONFIDENCE,
        'fallback_required': False,
    }

    for sl in SERVICE_LEVELS_SWEEP:
        record[
            f'safety_stock_q{int(sl * 100)}'
        ] = float(
            np.percentile(sim, sl * 100)
        )

    safety_stock_records_fold3.append(record)

safety_stock_df_fold3 = pd.DataFrame(
    safety_stock_records_fold3
)

assert (
    safety_stock_df_fold3['order_qty'] > 0
).all(), (
    'BUG: some Intermittent SKU has order_qty <= 0.'
)

assert safety_stock_df_fold3[
    'mean_lead_time_demand'
].notna().all(), (
    'BUG: intermittent mean lead-time demand contains NaN.'
)

assert safety_stock_df_fold3[
    'safety_stock_q80'
].notna().all(), (
    'BUG: intermittent q80 safety stock contains NaN.'
)

safety_stock_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/tsb_safety_stock_fold3.parquet'
)

print(
    f'Intermittent safety stock computed for '
    f'{len(safety_stock_df_fold3):,} SKUs '
    f'({safety_stock_df_fold3["low_confidence"].sum():,} '
    f'low-confidence, '
    f'{safety_stock_df_fold3["fallback_required"].sum():,} '
    f'fallback-required)'
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.4 — Unify Lumpy + Intermittent, then merge with Smooth/Erratic
# ══════════════════════════════════════════════════════════════════════════════════════

assert LEAD_TIME_DAYS == 7, (
    'Lumpy/Intermittent/Smooth-Erratic must share '
    'the same lead-time clock.'
)

lumpy_unified_fold3 = pd.DataFrame({
    'id': lumpy_policy_df_fold3['id'],
    'regime': 'lumpy',
    'method': 'historical_policy',
    'expected_lead_time_demand':
        lumpy_policy_df_fold3['avg_weekly'],
    'reorder_point':
        lumpy_policy_df_fold3['reorder_point'],
    'safety_buffer':
        lumpy_policy_df_fold3['reorder_point']
        - lumpy_policy_df_fold3['avg_weekly'],
    'order_qty':
        lumpy_policy_df_fold3['order_qty'],
    'low_confidence':
        lumpy_policy_df_fold3['max_weekly'] == 0,
    'fallback_required':
        lumpy_policy_df_fold3['max_weekly'] == 0,
})

intermittent_unified_fold3 = pd.DataFrame({
    'id': safety_stock_df_fold3['id'],
    'regime': 'intermittent',
    'method': 'block_bootstrap_tsb',
    'expected_lead_time_demand':
        safety_stock_df_fold3['mean_lead_time_demand'],
    'reorder_point':
        safety_stock_df_fold3['safety_stock_q80'],
    'safety_buffer':
        safety_stock_df_fold3['safety_stock_q80']
        - safety_stock_df_fold3['mean_lead_time_demand'],
    'order_qty':
        safety_stock_df_fold3['order_qty'],
    'low_confidence':
        safety_stock_df_fold3['low_confidence'],
    'fallback_required':
        safety_stock_df_fold3['fallback_required'],
})

lumpy_negative = lumpy_unified_fold3[
    lumpy_unified_fold3['safety_buffer'] < 0
]

assert len(lumpy_negative) == 0, (
    f'BUG: {len(lumpy_negative)} Lumpy SKUs '
    f'have negative safety_buffer.'
)

broken_reactive = intermittent_unified_fold3[
    (intermittent_unified_fold3['safety_buffer'] < 0)
    &
    (intermittent_unified_fold3['order_qty'] <= 0)
]

assert len(broken_reactive) == 0, (
    f'BUG: {len(broken_reactive)} Intermittent SKUs '
    f'would never reorder.'
)

lumpy_intermittent_fold3 = pd.concat(
    [
        lumpy_unified_fold3,
        intermittent_unified_fold3
    ],
    ignore_index=True
)

lumpy_intermittent_fold3[
    'sqrt_scaling_unvalidated'
] = False

lumpy_intermittent_fold3[
    'service_level'
] = 0.80

# Intermittent order_qty is already defined by the TSB policy
# in 5.3. Do not overwrite it during the merge.
assert (
    lumpy_intermittent_fold3.loc[
        lumpy_intermittent_fold3['regime'] == 'intermittent',
        'order_qty'
    ] > 0
).all(), (
    'BUG: some Intermittent order_qty <= 0 '
    'after regime merge.'
)

is_intermittent_f3 = lumpy_intermittent_fold3['regime'] == 'intermittent'
z_final_floor_f3 = lumpy_intermittent_fold3.loc[is_intermittent_f3, 'order_qty'].copy()
lumpy_intermittent_fold3.loc[is_intermittent_f3, 'order_qty'] = np.maximum(
    lumpy_intermittent_fold3.loc[is_intermittent_f3, 'expected_lead_time_demand'],
    z_final_floor_f3
).clip(lower=1.0)   # MIN_ORDER_ACTIVE_INTERMITTENT floor, 06f Section 6
print(f"Intermittent order_qty repatch: {is_intermittent_f3.sum():,} SKUs, "
      f"median {z_final_floor_f3.median():.2f} -> "
      f"{lumpy_intermittent_fold3.loc[is_intermittent_f3, 'order_qty'].median():.2f}")

final_reorder_params_fold3 = pd.concat(
    [
        lumpy_intermittent_fold3,
        smooth_erratic_unified_fold3
    ],
    ignore_index=True
)

assert final_reorder_params_fold3[
    'order_qty'
].notna().all() and (
    final_reorder_params_fold3['order_qty'] > 0
).all(), (
    'order_qty must be populated and positive '
    'across all regimes before simulation.'
)

assert final_reorder_params_fold3[
    'reorder_point'
].notna().all(), (
    'reorder_point must be populated across all regimes.'
)

final_reorder_params_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/final_reorder_params_fold3.parquet'
)

print(
    f'\nMerged Fold 3 reorder table: '
    f'{len(final_reorder_params_fold3):,} SKUs'
)

print(
    final_reorder_params_fold3['regime'].value_counts()
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.5 — Static simulation setup: real Fold 3 validation-window weekly actuals
# ══════════════════════════════════════════════════════════════════════════════════════

FOLD3_SIM_VAL_START = fold3_val['date'].min()
FOLD3_SIM_VAL_END = fold3_val['date'].max()

print(
    f'\nFold 3 val window for simulation: '
    f'{FOLD3_SIM_VAL_START.date()} -> '
    f'{FOLD3_SIM_VAL_END.date()}'
)

val_day_cols_fold3 = [
    d
    for d in day_cols_all
    if d in day_to_date
    and FOLD3_SIM_VAL_START <= day_to_date[d] <= FOLD3_SIM_VAL_END
]

val_dates_fold3 = pd.to_datetime([
    day_to_date[d]
    for d in val_day_cols_fold3
])

assert (
    len(val_dates_fold3) > 0
    and
    val_dates_fold3.max()
    >= FOLD3_SIM_VAL_END - pd.Timedelta(days=7)
), (
    'sales_train_validation.csv does not cover the Fold 3 '
    'val window -- swap in sales_train_evaluation.csv '
    'before trusting this simulation.'
)

print(
    f'Raw-sales coverage check passed: '
    f'{len(val_dates_fold3)} days found.'
)

sim_ids_fold3 = set(
    final_reorder_params_fold3['id']
)

sales_val_fold3 = (
    raw_sales[
        raw_sales['id'].isin(sim_ids_fold3)
    ]
    .set_index('id')[val_day_cols_fold3]
)

missing_sales_ids_fold3 = sorted(
    sim_ids_fold3 - set(sales_val_fold3.index)
)

print(
    f'SKUs with no matching row in raw sales file: '
    f'{len(missing_sales_ids_fold3):,}'
)

sales_val_t_fold3 = sales_val_fold3.T
sales_val_t_fold3.index = val_dates_fold3

weekly_actuals_fold3 = (
    sales_val_t_fold3
    .resample('W')
    .sum()
    .T
)

print(
    f'Weekly actual-demand matrix (Fold 3): '
    f'{weekly_actuals_fold3.shape[0]:,} SKUs x '
    f'{weekly_actuals_fold3.shape[1]} weeks'
)


def simulate_inventory(
    actual_weekly_demand,
    reorder_point,
    reorder_qty,
    initial_inventory,
    lead_time_weeks
):
    inventory = initial_inventory

    stockout_weeks = 0
    units_short = 0.0

    inventory_history = []

    pending_order = 0
    weeks_until_arrival = 0

    for week, demand in enumerate(actual_weekly_demand):

        if (
            weeks_until_arrival == 0
            and pending_order > 0
        ):
            inventory += pending_order
            pending_order = 0

        fulfilled = min(inventory, demand)

        if demand > inventory:
            stockout_weeks += 1
            units_short += demand - inventory

        inventory -= fulfilled

        if (
            inventory <= reorder_point
            and pending_order == 0
        ):
            pending_order = reorder_qty
            weeks_until_arrival = lead_time_weeks

        inventory_history.append(inventory)

        if weeks_until_arrival > 0:
            weeks_until_arrival -= 1

    return {
        'stockout_rate':
            stockout_weeks / len(actual_weekly_demand),
        'avg_inventory':
            np.mean(inventory_history),
        'fill_rate':
            1 - stockout_weeks / len(actual_weekly_demand),
        'units_short':
            units_short,
        'inventory_history':
            inventory_history,
    }


sim_rows = []
skipped_no_sales = 0

for row in final_reorder_params_fold3.itertuples(
    index=False
):
    sku_id = row.id

    if sku_id not in weekly_actuals_fold3.index:
        skipped_no_sales += 1
        continue

    demand = weekly_actuals_fold3.loc[
        sku_id
    ].values

    initial_inventory = (
        row.reorder_point + row.order_qty
    )

    result = simulate_inventory(
        demand,
        row.reorder_point,
        row.order_qty,
        initial_inventory,
        LEAD_TIME_WEEKS
    )

    sim_rows.append({
        'id': sku_id,
        'regime': row.regime,
        'service_level': row.service_level,
        **{
            k: v
            for k, v in result.items()
            if k != 'inventory_history'
        }
    })

simulation_results_fold3 = pd.DataFrame(sim_rows)

simulation_results_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/simulation_results_fold3.parquet'
)

print(
    f'\nSimulated {len(simulation_results_fold3):,} SKUs '
    f'({skipped_no_sales:,} skipped, no raw sales row).'
)

print(
    simulation_results_fold3
    .groupby('regime')[
        ['stockout_rate', 'avg_inventory', 'fill_rate']
    ]
    .describe()
    .round(3)
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.6 — Service-level sweep + naive baseline, Fold 3
# ══════════════════════════════════════════════════════════════════════════════════════

order_qty_by_id_fold3 = (
    final_reorder_params_fold3
    .set_index('id')['order_qty']
)

intermittent_safety_indexed_fold3 = (
    safety_stock_df_fold3
    .set_index('id')
)


def reorder_point_smooth_erratic_fold3(
    sku_id,
    sl
):
    regime = regime_lookup_fold3[sku_id]

    residuals = sku_residuals_fold3.get(
        sku_id,
        []
    )

    if len(residuals) < MIN_RESIDUALS_PER_SKU:
        resid_q = np.percentile(
            pooled_residuals_by_regime_fold3.get(
                regime,
                [0.0]
            ),
            sl * 100
        )
    else:
        resid_q = np.percentile(
            residuals,
            sl * 100
        )

    resid_q = max(resid_q, 0.0)

    pf_native = point_forecast_fold3.get(
        sku_id,
        0.0
    )

    return (
        pf_native * LEAD_TIME_DAYS
        + resid_q * np.sqrt(LEAD_TIME_DAYS)
    )


def reorder_point_intermittent_fold3(
    sku_id,
    sl
):
    return intermittent_safety_indexed_fold3.at[
        sku_id,
        f'safety_stock_q{int(sl * 100)}'
    ]


sweep_rows = []

for sl in SERVICE_LEVELS_SWEEP:

    sl_label = f'q{int(sl * 100)}'

    for row in final_reorder_params_fold3.itertuples(
        index=False
    ):

        sku_id = row.id
        regime = row.regime

        if sku_id not in weekly_actuals_fold3.index:
            continue

        if regime in ('smooth', 'erratic'):

            reorder_point = reorder_point_smooth_erratic_fold3(
                sku_id,
                sl
            )

        elif regime == 'intermittent':

            reorder_point = reorder_point_intermittent_fold3(
                sku_id,
                sl
            )

        else:

            reorder_point = row.reorder_point

        oq = order_qty_by_id_fold3[sku_id]

        result = simulate_inventory(
            weekly_actuals_fold3.loc[sku_id].values,
            reorder_point,
            oq,
            reorder_point + oq,
            LEAD_TIME_WEEKS
        )

        sweep_rows.append({
            'id': sku_id,
            'regime': regime,
            'service_level': sl_label,
            **{
                k: v
                for k, v in result.items()
                if k != 'inventory_history'
            }
        })

sweep_df_fold3 = pd.DataFrame(sweep_rows)

sweep_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/service_level_sweep_fold3.parquet'
)

print(
    '\n── Service-level sweep, pooled across regimes ──'
)

print(
    sweep_df_fold3
    .groupby('service_level')[
        [
            'stockout_rate',
            'avg_inventory',
            'fill_rate',
            'units_short'
        ]
    ]
    .mean()
    .round(3)
)


train_day_cols_fold3 = [
    d
    for d in day_cols_all
    if d in day_to_date
    and day_to_date[d] <= FOLD3_TRAIN_END_TS
]

naive_weekly_mean_fold3 = (
    raw_sales
    .set_index('id')[train_day_cols_fold3]
    .sum(axis=1)
    / (len(train_day_cols_fold3) / 7)
)

naive_rows = []

for sku_id, weekly_mean in naive_weekly_mean_fold3.items():

    if sku_id not in weekly_actuals_fold3.index:
        continue

    naive_qty = max(
        weekly_mean * LEAD_TIME_WEEKS,
        1.0
    )

    naive_rp = (
        weekly_mean * LEAD_TIME_WEEKS
    )

    result = simulate_inventory(
        weekly_actuals_fold3.loc[sku_id].values,
        naive_rp,
        naive_qty,
        naive_rp + naive_qty,
        LEAD_TIME_WEEKS
    )

    naive_rows.append({
        'id': sku_id,
        **{
            k: v
            for k, v in result.items()
            if k != 'inventory_history'
        }
    })

naive_df_fold3 = pd.DataFrame(
    naive_rows
)

naive_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/naive_baseline_fold3.parquet'
)

print(
    '\n── Naive baseline (Fold 3) ──'
)

print(
    naive_df_fold3[
        [
            'stockout_rate',
            'avg_inventory',
            'fill_rate',
            'units_short'
        ]
    ]
    .mean()
    .round(3)
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.7 — Real per-SKU unit cost, Fold 3 val window
# ══════════════════════════════════════════════════════════════════════════════════════

prices_raw = pd.read_csv(
    f'{RAW_DIR}/sell_prices.csv'
)

val_wm_yr_wk_fold3 = set(
    calendar_df[
        calendar_df['d'].isin(val_day_cols_fold3)
    ]['wm_yr_wk']
)

prices_val_fold3 = prices_raw[
    prices_raw['wm_yr_wk'].isin(
        val_wm_yr_wk_fold3
    )
]

unit_price_by_sku_fold3 = (
    prices_val_fold3
    .merge(
        raw_sales[
            ['id', 'item_id', 'store_id']
        ],
        on=['item_id', 'store_id'],
        how='inner'
    )
    .groupby(
        'id',
        observed=True
    )['sell_price']
    .mean()
)

fallback_price_fold3 = (
    unit_price_by_sku_fold3.median()
)

sweep_df_fold3['unit_cost'] = (
    sweep_df_fold3['id']
    .map(unit_price_by_sku_fold3)
    .fillna(fallback_price_fold3)
)

naive_df_fold3['unit_cost'] = (
    naive_df_fold3['id']
    .map(unit_price_by_sku_fold3)
    .fillna(fallback_price_fold3)
)

print(
    f'\nReal unit cost: '
    f'median=${unit_price_by_sku_fold3.median():.2f}, '
    f'fallback=${fallback_price_fold3:.2f}'
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.8 — Dynamic (periodic-review) policy, full population
# ══════════════════════════════════════════════════════════════════════════════════════

eligible_fold3 = (
    final_reorder_params_fold3[
        ~final_reorder_params_fold3['low_confidence']
    ]
    .copy()
)

se_ids_fold3 = eligible_fold3.loc[
    eligible_fold3['regime'].isin(
        ['smooth', 'erratic']
    ),
    'id'
].tolist()

mask_traj_fold3 = fold3_val['id'].isin(
    se_ids_fold3
)

traj_df_fold3 = (
    fold3_val[mask_traj_fold3]
    .sort_values(['id', 'date'])
    .copy()
)

traj_df_fold3['yhat'] = np.maximum(
    model_tweedie_fold3.predict(
        traj_df_fold3[FEATURE_COLS].values
    ),
    0
)

print(
    f'\nRebuilt traj_df for '
    f'{traj_df_fold3["id"].nunique():,} '
    f'Smooth/Erratic Fold 3 SKUs.'
)


def production_trajectory_fold3(
    sku_id,
    target_weeks
):
    sub = (
        traj_df_fold3[
            traj_df_fold3['id'] == sku_id
        ]
        .set_index('date')['yhat']
    )

    weekly = sub.resample('W').sum()

    return weekly.reindex(
        target_weeks,
        method='nearest'
    )


full_span_day_cols_fold3 = [
    d
    for d in day_cols_all
    if d in day_to_date
    and day_to_date[d] <= FOLD3_SIM_VAL_END
]

raw_sales_indexed = raw_sales.set_index('id')


def tsb_walkforward_weekly_fold3(
    sku_id,
    review_weeks,
    full_span_day_cols,
    day_to_date
):
    hist = (
        raw_sales_indexed.loc[
            sku_id,
            full_span_day_cols
        ]
        .values
        .astype(float)
    )

    dates = pd.to_datetime([
        day_to_date[d]
        for d in full_span_day_cols
    ])

    daily_series = pd.Series(
        hist,
        index=dates
    )

    review_dates = sorted(review_weeks)

    if len(review_dates) > 1:

        deltas = (
            pd.Series(review_dates[1:])
            - pd.Series(review_dates[:-1])
        )

        assert (
            deltas == pd.Timedelta(days=7)
        ).all(), (
            f'Irregular review spacing for '
            f'{sku_id}: {deltas.unique()}'
        )

    cutoff_to_week = {
        w - pd.Timedelta(days=6): w
        for w in review_dates
    }

    p, z, initialized = 0.0, 0.0, False
    forecasts = {}
    last_update_day = None

    for i, day in enumerate(
        daily_series.index
    ):

        occurred = (
            daily_series.iloc[i] > 0
        )

        if day in cutoff_to_week:

            week_label = cutoff_to_week[day]

            assert (
                last_update_day is None
                or last_update_day < day
            ), (
                f'{sku_id}: TSB state for week '
                f'{week_label} leaked -- '
                f'last updated {last_update_day}.'
            )

            forecasts[week_label] = (
                p * z
                if initialized
                else 0.0
            )

        if (
            not initialized
            and occurred
        ):

            z = daily_series.iloc[i]
            p = 1.0 / max(i + 1, 1)
            initialized = True
            last_update_day = day

        elif initialized:

            p = (
                BETA_LOCKED * occurred
                + (1 - BETA_LOCKED) * p
            )

            if occurred:
                z = (
                    ALPHA_LOCKED
                    * daily_series.iloc[i]
                    + (1 - ALPHA_LOCKED) * z
                )

            last_update_day = day

    weekly_fc = pd.Series({
        w: v * 7
        for w, v in forecasts.items()
    })

    return weekly_fc.reindex(
        review_dates,
        method='nearest'
    ).values


def get_dynamic_forecast_fold3(
    sku_id,
    regime,
    weeks
):

    if regime in (
        'smooth',
        'erratic'
    ):

        traj = (
            production_trajectory_fold3(
                sku_id,
                weeks
            )
            .reindex(weeks)
            .ffill()
            .bfill()
        )

        return traj.values, 'real'

    if regime == 'intermittent':

        return (
            tsb_walkforward_weekly_fold3(
                sku_id,
                weeks,
                full_span_day_cols_fold3,
                day_to_date
            ),
            'real'
        )

    static_val = (
        final_reorder_params_indexed_fold3
        .at[
            sku_id,
            'expected_lead_time_demand'
        ]
    )

    return (
        np.full(
            len(weeks),
            static_val
        ),
        'proxy'
    )


def dynamic_safety_stock(
    row,
    lead_time_weeks,
    review_period_weeks
):
    protection_weeks = (
        lead_time_weeks
        + review_period_weeks
    )

    scale = np.sqrt(
        protection_weeks
        / lead_time_weeks
    )

    return row['safety_buffer'] * scale


def simulate_inventory_periodic(
    actual_weekly_demand,
    forecast_weekly,
    safety_stock,
    lead_time_weeks,
    review_period_weeks,
    initial_inventory
):
    inventory = initial_inventory

    pipeline = [
        0.0
        for _ in range(lead_time_weeks)
    ]

    stockout_weeks = 0
    units_short = 0.0

    inventory_history = []

    protection_weeks = (
        lead_time_weeks
        + review_period_weeks
    )

    for week, demand in enumerate(
        actual_weekly_demand
    ):

        arriving = pipeline.pop(0)
        inventory += arriving
        pipeline.append(0.0)

        fulfilled = min(
            inventory,
            demand
        )

        if demand > inventory:
            stockout_weeks += 1
            units_short += (
                demand - inventory
            )

        inventory -= fulfilled

        target_demand = (
            forecast_weekly[week]
            if week < len(forecast_weekly)
            else forecast_weekly[-1]
        )

        order_up_to = (
            target_demand * protection_weeks
            + safety_stock
        )

        inventory_position = (
            inventory
            + sum(pipeline)
        )

        order_qty = max(
            order_up_to
            - inventory_position,
            0.0
        )

        pipeline[-1] = order_qty

        inventory_history.append(
            inventory
        )

    n = len(actual_weekly_demand)

    return {
        'stockout_rate':
            stockout_weeks / n,
        'avg_inventory':
            np.mean(inventory_history),
        'fill_rate':
            1 - stockout_weeks / n,
        'units_short':
            units_short,
    }


def naive_policy_fold3(sku_id):
    hist = raw_sales_indexed.loc[
        sku_id,
        train_day_cols_fold3
    ]

    weekly_mean = (
        hist.sum()
        / (len(train_day_cols_fold3) / 7)
    )

    return (
        max(weekly_mean, 0.0),
        max(weekly_mean, 1.0)
    )


final_reorder_params_indexed_fold3 = (
    final_reorder_params_fold3
    .set_index('id')
)


def compute_sku_metrics_full_fold3(
    sku_id,
    regime
):
    row = final_reorder_params_indexed_fold3.loc[
        sku_id
    ]

    demand = weekly_actuals_fold3.loc[
        sku_id
    ]

    weeks = demand.index

    unit_cost = unit_price_by_sku_fold3.get(
        sku_id,
        fallback_price_fold3
    )

    actual_total = demand.values.sum()

    naive_rp, naive_oq = naive_policy_fold3(
        sku_id
    )

    static_rp = row['reorder_point']
    static_oq = row['order_qty']

    naive_sim = simulate_inventory(
        demand.values,
        naive_rp,
        naive_oq,
        naive_rp + naive_oq,
        LEAD_TIME_WEEKS
    )

    static_sim = simulate_inventory(
        demand.values,
        static_rp,
        static_oq,
        static_rp + static_oq,
        LEAD_TIME_WEEKS
    )

    dyn_forecast, forecast_source = (
        get_dynamic_forecast_fold3(
            sku_id,
            regime,
            weeks
        )
    )

    dyn_safety = dynamic_safety_stock(
        row,
        LEAD_TIME_WEEKS,
        REVIEW_PERIOD_WEEKS
    )

    dyn_sim = simulate_inventory_periodic(
        demand.values,
        dyn_forecast,
        dyn_safety,
        LEAD_TIME_WEEKS,
        review_period_weeks=REVIEW_PERIOD_WEEKS,
        initial_inventory=static_rp + static_oq
    )

    dynamic_wape = (
        np.abs(
            demand.values
            - dyn_forecast
        ).sum()
        / actual_total
        if (
            forecast_source == 'real'
            and actual_total > 0
        )
        else np.nan
    )

    def total_cost(
        sim,
        n_weeks
    ):
        annualize = 52 / n_weeks

        return (
            sim['units_short']
            * unit_cost
            * STOCKOUT_MULT
            * annualize
            +
            sim['avg_inventory']
            * unit_cost
            * CARRY_RATE
        )

    naive_cost = total_cost(
        naive_sim,
        len(weeks)
    )

    static_cost = total_cost(
        static_sim,
        len(weeks)
    )

    dyn_cost = total_cost(
        dyn_sim,
        len(weeks)
    )

    static_vs_naive = (
        (naive_cost - static_cost)
        / naive_cost
        * 100
        if naive_cost
        else np.nan
    )

    dyn_vs_static = (
        (static_cost - dyn_cost)
        / static_cost
        * 100
        if static_cost
        else np.nan
    )

    annualize = 52 / len(weeks)

    return {
        'id': sku_id,
        'regime': regime,
        'method': row['method'],
        'dynamic_forecast_type': forecast_source,
        'zero_demand': bool(
            actual_total == 0
        ),
        'naive_cost_yr':
            round(naive_cost, 0),
        'static_cost_yr':
            round(static_cost, 0),
        'dynamic_cost_yr':
            round(dyn_cost, 0),
        'naive_fill':
            round(
                naive_sim['fill_rate'],
                3
            ),
        'static_fill':
            round(
                static_sim['fill_rate'],
                3
            ),
        'dynamic_fill':
            round(
                dyn_sim['fill_rate'],
                3
            ),
        'naive_stockout_rate':
            round(
                naive_sim['stockout_rate'],
                3
            ),
        'static_stockout_rate':
            round(
                static_sim['stockout_rate'],
                3
            ),
        'dynamic_stockout_rate':
            round(
                dyn_sim['stockout_rate'],
                3
            ),
        'naive_avg_inventory':
            round(
                naive_sim['avg_inventory'],
                3
            ),
        'static_avg_inventory':
            round(
                static_sim['avg_inventory'],
                3
            ),
        'dynamic_avg_inventory':
            round(
                dyn_sim['avg_inventory'],
                3
            ),
        'static_vs_naive_%':
            round(
                static_vs_naive,
                1
            ),
        'dynamic_vs_static_%':
            round(
                dyn_vs_static,
                1
            ),
        'dynamic_wape':
            dynamic_wape,
        'unit_cost':
            round(unit_cost, 4),
        'naive_units_short_annualized':
            round(
                naive_sim['units_short']
                * annualize,
                3
            ),
        'static_units_short_annualized':
            round(
                static_sim['units_short']
                * annualize,
                3
            ),
        'dynamic_units_short_annualized':
            round(
                dyn_sim['units_short']
                * annualize,
                3
            ),
    }


sampled_ids_full_fold3 = {
    regime: eligible_fold3.loc[
        eligible_fold3['regime'] == regime,
        'id'
    ].tolist()
    for regime in [
        'smooth',
        'erratic',
        'intermittent',
        'lumpy'
    ]
}

for regime, ids in sampled_ids_full_fold3.items():
    print(
        f'  {regime}: {len(ids):,} eligible SKUs'
    )

full_results_fold3 = []

total_skus = sum(
    len(v)
    for v in sampled_ids_full_fold3.values()
)

done = 0
t0 = time.time()

for regime, ids in sampled_ids_full_fold3.items():

    print(
        f'\nStarting {regime} '
        f'({len(ids):,} SKUs)...'
    )

    for sid in ids:

        if sid not in weekly_actuals_fold3.index:
            continue

        full_results_fold3.append(
            compute_sku_metrics_full_fold3(
                sid,
                regime
            )
        )

        done += 1

        if done % 2000 == 0:

            elapsed = (
                time.time() - t0
            )

            rate = (
                done / elapsed
            )

            remaining = (
                total_skus - done
            ) / rate

            print(
                f'  {done:,}/{total_skus:,} done '
                f'({elapsed / 60:.1f} min elapsed, '
                f'~{remaining / 60:.1f} min remaining)'
            )


full_summary_df_fold3 = pd.DataFrame(
    full_results_fold3
)

clean_df_fold3 = (
    full_summary_df_fold3[
        ~full_summary_df_fold3['zero_demand']
    ]
    .copy()
)

clean_df_fold3['dynamic_wins'] = (
    clean_df_fold3['dynamic_cost_yr']
    < clean_df_fold3['static_cost_yr']
)

clean_df_fold3['static_wins_vs_naive'] = (
    clean_df_fold3['static_cost_yr']
    < clean_df_fold3['naive_cost_yr']
)


def q25(s):
    return s.quantile(0.25)


def q75(s):
    return s.quantile(0.75)


static_vs_naive_summary_fold3 = (
    clean_df_fold3
    .groupby('regime')
    .agg(
        n_skus=('id', 'count'),
        win_rate_pct=(
            'static_wins_vs_naive',
            lambda s: round(
                s.mean() * 100,
                1
            )
        ),
        median_static_vs_naive_pct=(
            'static_vs_naive_%',
            'median'
        ),
        iqr25_static_vs_naive_pct=(
            'static_vs_naive_%',
            q25
        ),
        iqr75_static_vs_naive_pct=(
            'static_vs_naive_%',
            q75
        ),
    )
    .round(3)
    .reset_index()
)

regime_summary_fold3 = (
    clean_df_fold3
    .groupby('regime')
    .agg(
        n_skus=('id', 'count'),
        win_rate_pct=(
            'dynamic_wins',
            lambda s: round(
                s.mean() * 100,
                1
            )
        ),
        median_dyn_vs_static_pct=(
            'dynamic_vs_static_%',
            'median'
        ),
        iqr25_dyn_vs_static_pct=(
            'dynamic_vs_static_%',
            q25
        ),
        iqr75_dyn_vs_static_pct=(
            'dynamic_vs_static_%',
            q75
        ),
        median_wape=(
            'dynamic_wape',
            'median'
        ),
        p90_wape=(
            'dynamic_wape',
            lambda s: s.quantile(0.90)
        ),
        n_real_forecast=(
            'dynamic_forecast_type',
            lambda s: (
                s == 'real'
            ).sum()
        ),
        mean_static_stockout_rate=(
            'static_stockout_rate',
            'mean'
        ),
        mean_dynamic_stockout_rate=(
            'dynamic_stockout_rate',
            'mean'
        ),
        mean_static_avg_inventory=(
            'static_avg_inventory',
            'mean'
        ),
        mean_dynamic_avg_inventory=(
            'dynamic_avg_inventory',
            'mean'
        ),
    )
    .round(3)
    .reset_index()
)

pooled_fold3 = (
    clean_df_fold3
    .groupby('regime')[
        [
            'naive_cost_yr',
            'static_cost_yr',
            'dynamic_cost_yr'
        ]
    ]
    .sum()
    .reset_index()
)

pooled_fold3[
    'static_vs_naive_%'
] = (
    (
        pooled_fold3['naive_cost_yr']
        - pooled_fold3['static_cost_yr']
    )
    / pooled_fold3['naive_cost_yr']
    * 100
)

pooled_fold3[
    'dynamic_vs_static_%'
] = (
    (
        pooled_fold3['static_cost_yr']
        - pooled_fold3['dynamic_cost_yr']
    )
    / pooled_fold3['static_cost_yr']
    * 100
)

pooled_fold3['n_skus'] = (
    clean_df_fold3
    .groupby('regime')
    .size()
    .reindex(
        pooled_fold3['regime']
    )
    .values
)

print(
    '\n══ Static vs. Naive — '
    'Fold 3, full population ══'
)

print(
    static_vs_naive_summary_fold3
    .to_string(index=False)
)

print(
    '\n══ Dynamic vs. Static — '
    'Fold 3, per-regime ══'
)

print(
    regime_summary_fold3
    .to_string(index=False)
)

print(
    '\n══ Pooled-dollar regime summary, '
    'Fold 3 ══'
)

print(
    pooled_fold3
    .round(1)
    .to_string(index=False)
)

g = pooled_fold3[
    [
        'naive_cost_yr',
        'static_cost_yr',
        'dynamic_cost_yr'
    ]
].sum()

print(
    f"\nHeadline (Fold 3): dynamic vs static "
    f"{(g['static_cost_yr'] - g['dynamic_cost_yr']) / g['static_cost_yr'] * 100:+.1f}% | "
    f"dynamic vs naive "
    f"{(g['naive_cost_yr'] - g['dynamic_cost_yr']) / g['naive_cost_yr'] * 100:+.1f}% "
    f"({clean_df_fold3.shape[0]:,} SKUs, "
    f"{full_summary_df_fold3['zero_demand'].sum():,} zero-demand excluded)"
)

full_summary_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/dynamic_policy_full_population_fold3.parquet'
)

regime_summary_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/dynamic_policy_regime_summary_fold3.parquet'
)

pooled_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/dynamic_policy_pooled_fold3.parquet'
)

static_vs_naive_summary_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/dynamic_policy_static_vs_naive_fold3.parquet'
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.9 — Dynamic cost sensitivity
# ══════════════════════════════════════════════════════════════════════════════════════

cost_base_fold3 = clean_df_fold3[
    [
        'regime',
        'naive_units_short_annualized',
        'static_units_short_annualized',
        'dynamic_units_short_annualized',
        'naive_avg_inventory',
        'static_avg_inventory',
        'dynamic_avg_inventory',
        'unit_cost',
    ]
].copy()

s8_rows = []

for stockout_mult in STOCKOUT_MULTIPLIERS:

    for carry_rate in CARRYING_RATES:

        naive_cost = (
            cost_base_fold3[
                'naive_units_short_annualized'
            ]
            * cost_base_fold3['unit_cost']
            * stockout_mult
            +
            cost_base_fold3[
                'naive_avg_inventory'
            ]
            * cost_base_fold3['unit_cost']
            * carry_rate
        ).sum()

        static_cost = (
            cost_base_fold3[
                'static_units_short_annualized'
            ]
            * cost_base_fold3['unit_cost']
            * stockout_mult
            +
            cost_base_fold3[
                'static_avg_inventory'
            ]
            * cost_base_fold3['unit_cost']
            * carry_rate
        ).sum()

        dynamic_cost = (
            cost_base_fold3[
                'dynamic_units_short_annualized'
            ]
            * cost_base_fold3['unit_cost']
            * stockout_mult
            +
            cost_base_fold3[
                'dynamic_avg_inventory'
            ]
            * cost_base_fold3['unit_cost']
            * carry_rate
        ).sum()

        s8_rows.append({
            'stockout_mult': stockout_mult,
            'carry_rate': carry_rate,
            'ratio': round(
                stockout_mult / carry_rate,
                1
            ),
            'naive_cost_$':
                round(naive_cost, 0),
            'static_cost_$':
                round(static_cost, 0),
            'dynamic_cost_$':
                round(dynamic_cost, 0),
            'dynamic_vs_static_%':
                round(
                    (static_cost - dynamic_cost)
                    / static_cost
                    * 100,
                    1
                ),
            'dynamic_vs_naive_%':
                round(
                    (naive_cost - dynamic_cost)
                    / naive_cost
                    * 100,
                    1
                ),
        })

dynamic_sensitivity_df_fold3 = pd.DataFrame(
    s8_rows
)

dynamic_sensitivity_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/dynamic_cost_sensitivity_fold3.parquet'
)

print(
    '\n══ Dynamic cost sensitivity, '
    'Fold 3 (24 scenarios) ══'
)

print(
    dynamic_sensitivity_df_fold3
    .to_string(index=False)
)


# ══════════════════════════════════════════════════════════════════════════════════════
# 5.10 — Head-to-head: static (own cost-minimizing level) vs dynamic
# ══════════════════════════════════════════════════════════════════════════════════════

population_ids_fold3 = set(
    clean_df_fold3['id']
)

sweep_df_scoped_fold3 = sweep_df_fold3[
    sweep_df_fold3['id'].isin(
        population_ids_fold3
    )
].copy()

naive_df_scoped_fold3 = naive_df_fold3[
    naive_df_fold3['id'].isin(
        population_ids_fold3
    )
].copy()

assert (
    sweep_df_scoped_fold3['id'].nunique()
    ==
    naive_df_scoped_fold3['id'].nunique()
    ==
    len(population_ids_fold3)
), (
    'Scoped sweep_df/naive_df SKU counts '
    'do not match clean_df_fold3 -- investigate.'
)

N_WEEKS_VAL_FOLD3 = (
    weekly_actuals_fold3.shape[1]
)

ANNUALIZE_S9_FOLD3 = (
    52 / N_WEEKS_VAL_FOLD3
)

print(
    f'\nFold 3 val window: '
    f'{N_WEEKS_VAL_FOLD3} weeks -> '
    f'annualize factor '
    f'{ANNUALIZE_S9_FOLD3:.4f}'
)

s9_rows = []

for stockout_mult in STOCKOUT_MULTIPLIERS:

    for carry_rate in CARRYING_RATES:

        stockout_cost = (
            sweep_df_scoped_fold3[
                'units_short'
            ]
            * ANNUALIZE_S9_FOLD3
            * sweep_df_scoped_fold3[
                'unit_cost'
            ]
            * stockout_mult
        )

        carry_cost = (
            sweep_df_scoped_fold3[
                'avg_inventory'
            ]
            * sweep_df_scoped_fold3[
                'unit_cost'
            ]
            * carry_rate
        )

        total_by_level = (
            stockout_cost + carry_cost
        ).groupby(
            sweep_df_scoped_fold3[
                'service_level'
            ]
        ).sum().reindex(SL_ORDER)

        static_optimal_level = (
            total_by_level.idxmin()
        )

        static_optimal_cost = (
            total_by_level.min()
        )

        naive_total = (
            naive_df_scoped_fold3[
                'units_short'
            ]
            * ANNUALIZE_S9_FOLD3
            * naive_df_scoped_fold3[
                'unit_cost'
            ]
            * stockout_mult
            +
            naive_df_scoped_fold3[
                'avg_inventory'
            ]
            * naive_df_scoped_fold3[
                'unit_cost'
            ]
            * carry_rate
        ).sum()

        dyn_row = dynamic_sensitivity_df_fold3[
            (
                dynamic_sensitivity_df_fold3[
                    'stockout_mult'
                ]
                == stockout_mult
            )
            &
            (
                dynamic_sensitivity_df_fold3[
                    'carry_rate'
                ]
                == carry_rate
            )
        ].iloc[0]

        dynamic_cost = (
            dyn_row['dynamic_cost_$']
        )

        s9_rows.append({
            'stockout_mult':
                stockout_mult,
            'carry_rate':
                carry_rate,
            'ratio':
                round(
                    stockout_mult / carry_rate,
                    1
                ),
            'static_optimal_level':
                static_optimal_level,
            'naive_cost_$':
                round(naive_total, 0),
            'static_optimal_cost_$':
                round(static_optimal_cost, 0),
            'dynamic_cost_$':
                round(dynamic_cost, 0),
            'dynamic_vs_static_optimal_%':
                round(
                    (
                        static_optimal_cost
                        - dynamic_cost
                    )
                    / static_optimal_cost
                    * 100,
                    1
                ),
            'static_optimal_vs_naive_%':
                round(
                    (
                        naive_total
                        - static_optimal_cost
                    )
                    / naive_total
                    * 100,
                    1
                ),
        })

headtohead_df_fold3 = pd.DataFrame(
    s9_rows
)

headtohead_df_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/static_vs_dynamic_headtohead_fold3.parquet'
)

print(
    '\n══ Head-to-head: static '
    '(own optimal level) vs dynamic, Fold 3 ══'
)

print(
    headtohead_df_fold3
    .to_string(index=False)
)

print(
    f"\nStatic's chosen level by scenario: "
    f"{headtohead_df_fold3['static_optimal_level'].value_counts().to_dict()}"
)

print(
    f"Dynamic beats static's OWN best level in "
    f"{(headtohead_df_fold3['dynamic_vs_static_optimal_%'] > 0).sum()}/24 scenarios, "
    f"range "
    f"{headtohead_df_fold3['dynamic_vs_static_optimal_%'].min():.1f}% "
    f"to "
    f"{headtohead_df_fold3['dynamic_vs_static_optimal_%'].max():.1f}%."
)

print(
    '\n✓ Section 5 complete. '
    'All Fold 3 artifacts saved with _fold3 suffix in predictions/.'
)

Smooth/Erratic Fold 3 reorder table: 8,895 SKUs (122 excluded, no monitor-window data; 0 NaN-guarded)

Dense raw Fold 3 demand history: 1,460 days (2011-02-02 -> 2015-01-31)

Lumpy SKUs (Fold 3): 4,170
Lumpy policy computed for 4,170 SKUs (665 zero-demand-in-training, floored at MIN_ORDER_ZERO=5)

Intermittent SKUs (Fold 3): 17,303
Loaded locked TSB params (06f, not re-derived): alpha=0.5, beta=0.5
Dense intermittent demand matrix: 17,303 SKUs x 1,460 days
Intermittent safety stock computed for 17,303 SKUs (0 low-confidence, 0 fallback-required)
Intermittent order_qty repatch: 17,303 SKUs, median 1.29 -> 2.54

Merged Fold 3 reorder table: 30,368 SKUs
regime
intermittent    17303
smooth           8003
lumpy            4170
erratic           892
Name: count, dtype: int64

Fold 3 val window for simulation: 2015-02-01 -> 2016-01-31
Raw-sales coverage check passed: 365 days found.
SKUs with no matching row in raw sales file: 0
Weekly actual-demand matrix (Fold 3): 30,368 SKUs x 53 weeks

Si

#### Section 5 Findings — Full Pipeline Run on Fold 3 Validation

**Status: LOCKED** (re-run once after a methodology fix — see below).

**Re-run disclosure.** The first execution omitted 06g Section 1's Intermittent `order_qty`
repatch, leaving `order_qty = max(z_final, 1)` instead of
`max(expected_lead_time_demand, z_final, 1)`. That under-ordered on 57% of the population,
crippling the static baseline and inflating every dynamic-vs-static number. Fixed and re-run.
This corrects a divergence from the locked Fold 2 method; it is not selection against Fold 3
results, and the pre-fix numbers are superseded, not suppressed. Sections 3 and 4 were
untouched and remained valid. Post-fix repatch: median 1.29 → 2.54 (Fold 2: 1.36 → 3.04).

**Run integrity.** 365 days of raw-sales coverage, 0 SKUs missing a sales row, 30,368 SKUs
simulated with 0 skipped, 122 Smooth/Erratic excluded for no monitor-window data (matches
Section 4 exactly), 0 low-confidence or fallback-required on Intermittent. Annualization
0.9811 on the stockout term only, per 06g Section 9's corrected convention.

**Headline:** dynamic beats static's own cost-minimizing level in 24/24 scenarios,
**87.5%–91.8%**; static picks q99 in all 24, no interior minimum — same "best point in the
tested q50–q99 range, not a confirmed optimum" caveat as 06g. Dynamic vs. naive 93.2%–95.4%.

**Fold 2 comparison — what held:**

| | Fold 2 | Fold 3 |
|---|---|---|
| Median WAPE (smooth / interm / erratic) | .376 / .907 / .425 | .365 / .879 / .425 |
| p90 WAPE | .638 / 1.596 / .742 | .646 / 1.546 / .747 |
| Static-beats-naive win rate, Intermittent | 77.1% | 76.7% |
| Dyn-vs-static win rate, Lumpy | 35.5% | 32.2% |
| Static's cost-minimizing level | q99, 24/24 | q99, 24/24 |

Forecast accuracy is effectively identical on an unseen year — that is the load-bearing
result, and it is stable. Intermittent's static win rate landing within 0.4pp of Fold 2 is
the strongest confirmation the repatch fix restored the right baseline.

**Fold 2 comparison — what moved, and why:**

- **Regime mix shifted materially**: Intermittent 46.9%→57.0%, Lumpy 23.0%→13.7% (Smooth and
  Erratic stable within 1pp). Per Section 2, this is plausibly a training-window-length
  artifact, not confirmed drift.
- **Pooled q80 stockout 0.218→0.281, fill 0.782→0.719.** Both outside the plan's ±5pp
  tolerance. But the naive baseline degraded in the same direction and by a similar amount
  (stockout 0.356→0.412, fill 0.644→0.588) on less inventory (4.924→4.581), so Fold 3's
  validation year is harder in absolute terms rather than the policy having broken. Relative
  standing vs. naive is preserved.
- **Dynamic vs. static-optimal rose 79.3–85.6% → 87.5–91.8%** (+7.4pp at the 2.0x/25%
  default). This tracks the regime mix directly: dynamic's edge is largest on Intermittent
  (92.4% pooled-dollar) and near-zero on Lumpy (12.9%), so growing the former and shrinking
  the latter raises the pooled number without any change in forecast skill. Report the
  improvement as mix-driven, not as evidence the model got better.
- **Static-optimal vs. naive fell 51.7–53.9% → 42.0–43.4%.** Same mechanism in reverse:
  Lumpy is the regime where static most outperforms naive (+81.0%), and there is less of it.

**Open item for Section 6/7, not blocking:** pooled-dollar static-vs-naive flips sign for
Smooth (+12.7%→−28.5%) and Erratic (+15.4%→−47.7%) between folds, while the per-SKU median
stays negative in both (−17.9%→−34.2%, −20.9%→−79.0%) and win rates move only modestly
(44.6%→39.6%, 41.5%→31.0%). A pooled-dollar sign flip without a comparable win-rate flip
means a small number of high-value SKUs dominate the aggregate. Worth one sentence in the
writeup; does not affect the headline, which is computed on the full population.

**Unchanged by design:** Lumpy's `n_real_forecast = 0` and NaN WAPE — Lumpy routes to the
historical min-max policy and never consumes the ML forecast, same as Fold 2.

Artifacts saved with `_fold3` suffix. Valid input to Section 6.

## Section 6 — Evaluate Against Fold 2 Benchmarks

Fold 2 values are constants read from the locked 06e/06f/06g outputs — not recomputed here.
Fold 3 values come from Section 5's in-memory results. Tolerances are the plan's, fixed
before the numbers were seen. This section decides whether Fold 3 behaves like a normal
next-year run or like a distribution shift, per metric rather than in aggregate.

In [25]:
# -- 07 Section 6: Fold 3 vs Fold 2 Stability Check -------------------------------------
# Pure comparison. No simulation, no model calls, no additional Fold 3 touch.

# ── Fold 2 benchmarks — LOCKED CONSTANTS, provenance in comments ──────────────────────
F2 = {
    'n_skus':            30442,                # 06g Section 1 merged table
    'pct_smooth':        8344 / 30442 * 100,   # 06g Section 1 regime value_counts
    'pct_intermittent': 14268 / 30442 * 100,
    'pct_lumpy':         7003 / 30442 * 100,
    'pct_erratic':        827 / 30442 * 100,
    'median_wape_smooth':       0.376,         # 06g Section 7 full-population metrics
    'median_wape_intermittent': 0.907,
    'median_wape_erratic':      0.425,
    'p90_wape_smooth':          0.638,
    'p90_wape_intermittent':    1.596,
    'p90_wape_erratic':         0.742,
    'stockout_q80':      0.218,                # 06g cell 9, pooled sweep across regimes
    'dyn_vs_static_opt_default': 81.8,         # 06g Section 9, 2.0x / 25%
}
# q90 empirical coverage, Fold 2: 0.899 (06e, held-out coverage-test split, 1,043,717 rows).
# Not comparable to anything Fold 3 can produce here — see note below. Left unset.
F2_Q90_COVERAGE = None

DEFAULT_MULT, DEFAULT_CARRY = 2.0, 0.25

# ── Fold 3 values, pulled from Section 5's live objects ───────────────────────────────
n3 = len(sku_regimes_fold3)
rc3 = sku_regimes_fold3['regime'].value_counts()

# regime_summary_fold3 is the aggregated per-regime table (median_wape/p90_wape columns),
# built from clean_df_fold3 with zero-demand SKUs excluded — same construction as 06g's
# regime_summary, which is where the F2 WAPE constants above came from.
wape3 = regime_summary_fold3.set_index('regime')

# Pooled-across-regime sweep table — mirrors 06g cell 9's printed "Pooled across all
# regimes (headline table)" (sweep_df.groupby('service_level').mean()). Unscoped, matching
# how the F2 stockout_q80 constant was computed (sweep_df_scoped_fold3 would not match).
pooled_sweep_fold3 = (
    sweep_df_fold3
    .groupby('service_level')[['stockout_rate', 'avg_inventory', 'fill_rate', 'units_short']]
    .mean()
)
pooled_q80 = pooled_sweep_fold3.loc['q80']

h2h_default = headtohead_df_fold3[
    (headtohead_df_fold3['stockout_mult'] == DEFAULT_MULT) &
    (headtohead_df_fold3['carry_rate'] == DEFAULT_CARRY)
].iloc[0]

# NOTE: no cov3 computation here. 07 Section 4 deliberately calibrates on ALL of
# monitor_sub with no calib/test split (see Section 4 markdown) — any percentile-vs-same-
# vector coverage check is tautological (~0.90 by construction, always "passes") and would
# misreport an unvalidated interval as validated. Skipped; disclosed in Findings instead.

# ── Comparison table ─────────────────────────────────────────────────────────────────
# Row tuples: (label, fold2, fold3, tolerance, unit, scale)
# unit: 'pct' = percentage-point delta, 'rel' = relative % delta
# scale: explicit multiplier to put fold2/fold3 on a 0-100 percentage-point basis for
# 'pct' rows (rather than inferring it from magnitude, which breaks once a fold3 value
# for a naturally-<1 quantity crosses below 1.0).
rows = [
    ('Regime distribution (% Smooth)', F2['pct_smooth'],
     rc3.get('smooth', 0) / n3 * 100, 5.0, 'pct', 1.0),
    ('Regime distribution (% Intermittent)', F2['pct_intermittent'],
     rc3.get('intermittent', 0) / n3 * 100, 5.0, 'pct', 1.0),
    ('Regime distribution (% Lumpy)', F2['pct_lumpy'],
     rc3.get('lumpy', 0) / n3 * 100, 5.0, 'pct', 1.0),
    ('Regime distribution (% Erratic)', F2['pct_erratic'],
     rc3.get('erratic', 0) / n3 * 100, 5.0, 'pct', 1.0),
    ('Median WAPE — Smooth', F2['median_wape_smooth'],
     wape3.loc['smooth', 'median_wape'], 5.0, 'rel', None),
    ('Median WAPE — Intermittent', F2['median_wape_intermittent'],
     wape3.loc['intermittent', 'median_wape'], 5.0, 'rel', None),
    ('Median WAPE — Erratic', F2['median_wape_erratic'],
     wape3.loc['erratic', 'median_wape'], 5.0, 'rel', None),
    ('p90 WAPE — Smooth', F2['p90_wape_smooth'],
     wape3.loc['smooth', 'p90_wape'], 10.0, 'rel', None),
    ('p90 WAPE — Intermittent', F2['p90_wape_intermittent'],
     wape3.loc['intermittent', 'p90_wape'], 10.0, 'rel', None),
    ('p90 WAPE — Erratic', F2['p90_wape_erratic'],
     wape3.loc['erratic', 'p90_wape'], 10.0, 'rel', None),
    ('Simulated stockout rate (q80)', F2['stockout_q80'],
     pooled_q80['stockout_rate'], 5.0, 'pct', 100.0),
    # Fill rate intentionally omitted: fill_rate = 1 - stockout_rate in this simulator,
    # so it is the complement of the row above and would double-count the same signal.
    ('Dyn vs static-optimal (2.0x/25%)', F2['dyn_vs_static_opt_default'],
     h2h_default['dynamic_vs_static_optimal_%'], 5.0, 'pct', 1.0),
]

if F2_Q90_COVERAGE is not None:
    rows.insert(10, ('q90 empirical coverage', F2_Q90_COVERAGE, None, 3.0, 'pct', 100.0))

recs = []
for label, f2, f3, tol, unit, scale in rows:
    if unit == 'pct':
        delta = (f3 - f2) * scale
        stable = abs(delta) <= tol
        disp = f'{delta:+.1f} pp'
    else:
        delta = (f3 - f2) / f2 * 100 if f2 else np.nan
        stable = abs(delta) <= tol
        disp = f'{delta:+.1f} %'
    recs.append({
        'Metric': label,
        'Fold 2': round(f2, 3),
        'Fold 3': round(f3, 3),
        'Delta': disp,
        'Tolerance': f'±{tol:g}{" pp" if unit == "pct" else " %"}',
        'Stable?': 'STABLE' if stable else 'UNSTABLE',
    })

stability_df = pd.DataFrame(recs)

print('══ Fold 3 vs Fold 2 Stability Check ══\n')
print(stability_df.to_string(index=False))

n_unstable = (stability_df['Stable?'] == 'UNSTABLE').sum()
print(f'\n{n_unstable} / {len(stability_df)} metrics outside tolerance.')

# ── Forecast-side vs policy-side vs mix split — the plan's "flag, don't average" ──────
mix_rows = stability_df['Metric'].str.contains('Regime distribution')
forecast_rows = stability_df['Metric'].str.contains('WAPE|coverage') & ~mix_rows
policy_rows = stability_df['Metric'].str.contains('stockout|fill|Dyn vs') & ~mix_rows

print(f"\nRegime-mix metrics unstable    : "
      f"{(stability_df.loc[mix_rows, 'Stable?'] == 'UNSTABLE').sum()}"
      f" / {mix_rows.sum()}")
print(f"Forecast-side metrics unstable : "
      f"{(stability_df.loc[forecast_rows, 'Stable?'] == 'UNSTABLE').sum()}"
      f" / {forecast_rows.sum()}")
print(f"Policy-side metrics unstable   : "
      f"{(stability_df.loc[policy_rows, 'Stable?'] == 'UNSTABLE').sum()}"
      f" / {policy_rows.sum()}")
print('\nForecast-side stable, mix shifted, policy-side (stockout, dyn-vs-static) unstable '
      'points at the regime-mix shift propagating through the inventory policy, not at '
      'model drift — see Section 5 Findings and the Fold 3 caveat (Intermittent '
      '46.9%->57.0%, Lumpy 23.0%->13.7%).')

stability_df.to_parquet(f'{PREDICTIONS_DIR}/fold3_stability_check.parquet')
print(f'\n✓ Saved: fold3_stability_check.parquet')

══ Fold 3 vs Fold 2 Stability Check ══

                              Metric  Fold 2  Fold 3   Delta Tolerance  Stable?
      Regime distribution (% Smooth)  27.410  26.602 -0.8 pp     ±5 pp   STABLE
Regime distribution (% Intermittent)  46.869  56.750 +9.9 pp     ±5 pp UNSTABLE
       Regime distribution (% Lumpy)  23.004  13.677 -9.3 pp     ±5 pp UNSTABLE
     Regime distribution (% Erratic)   2.717   2.971 +0.3 pp     ±5 pp   STABLE
                Median WAPE — Smooth   0.376   0.365  -2.9 %      ±5 %   STABLE
          Median WAPE — Intermittent   0.907   0.879  -3.1 %      ±5 %   STABLE
               Median WAPE — Erratic   0.425   0.425  +0.0 %      ±5 %   STABLE
                   p90 WAPE — Smooth   0.638   0.646  +1.3 %     ±10 %   STABLE
             p90 WAPE — Intermittent   1.596   1.546  -3.1 %     ±10 %   STABLE
                  p90 WAPE — Erratic   0.742   0.747  +0.7 %     ±10 %   STABLE
       Simulated stockout rate (q80)   0.218   0.281 +6.3 pp     ±5 pp UNSTABLE


## Section 6 — Evaluate Against Fold 2 Benchmarks

Fold 3 is compared against the locked Fold 2 benchmarks using tolerances fixed before the comparison. This section is diagnostic only — **no model, threshold, service level, or policy is changed based on Fold 3 results.**

### Stability Check

| Metric                     | Fold 2 | Fold 3 |       Delta | Status       |
| -------------------------- | -----: | -----: | ----------: | ------------ |
| Regime — Smooth            |  27.4% |  26.6% |     −0.8 pp | STABLE       |
| Regime — Intermittent      |  46.9% |  56.7% | **+9.8 pp** | **UNSTABLE** |
| Regime — Lumpy             |  23.0% |  13.7% | **−9.3 pp** | **UNSTABLE** |
| Regime — Erratic           |   2.7% |   3.0% |     +0.3 pp | STABLE       |
| Median WAPE — Smooth       |  0.376 |  0.365 |       −2.9% | STABLE       |
| Median WAPE — Intermittent |  0.907 |  0.879 |       −3.1% | STABLE       |
| Median WAPE — Erratic      |  0.425 |  0.425 |        0.0% | STABLE       |
| p90 WAPE — Smooth          |  0.638 |  0.646 |       +1.3% | STABLE       |
| p90 WAPE — Intermittent    |  1.596 |  1.546 |       −3.1% | STABLE       |
| p90 WAPE — Erratic         |  0.742 |  0.747 |       +0.7% | STABLE       |
| q80 Stockout Rate          |  0.218 |  0.281 | **+6.3 pp** | **UNSTABLE** |
| Dynamic vs Static-Optimal  |  81.8% | ~89.2% | **+7.4 pp** | **UNSTABLE** |

**4 / 12 metrics** are outside tolerance.

### What this means

**Forecasting is stable.** All six WAPE metrics stayed within tolerance, including Intermittent WAPE despite Intermittent becoming a much larger share of the population. There is no evidence from Fold 3 that the frozen forecasting model needs to be improved or retuned.

**The main shift is regime composition.** Intermittent grew from 46.9% to 56.7%, while Lumpy fell from 23.0% to 13.7%. Section 2 showed that this is dominated by Lumpy → Intermittent reclassification. The longer Fold 3 training window is a plausible contributor, so this should not be interpreted as confirmed demand drift.

**Inventory metrics changed, but not in isolation.** q80 stockout rate increased from 0.218 to 0.281, but the naive baseline also became worse (0.356 → 0.412). This indicates that Fold 3 is a harder validation environment and that the change is not simply evidence that the proposed policy stopped working.

**Dynamic vs. static-optimal improved from 81.8% to ~89.2%.** Section 5 indicates that this is largely associated with the changed regime mix: Intermittent now represents substantially more of the population, and dynamic policy performs relatively better for that regime.

### Do we need to improve anything?

**No changes should be made from the Fold 3 results.** The forecasting side is stable, and Fold 3 is explicitly a one-shot production simulation rather than another tuning opportunity.

The only item worth carrying forward is the **regime-classification/data-quality caveat**: `fold3_train` was found to omit real positive-demand observations, so Section 5's Lumpy and Intermittent demand histories were corrected to use the dense raw M5 panel. The already-computed Section 2 regime assignments are not retroactively changed.

### Conclusion

**Fold 3 passes the main generalization test:** forecast accuracy remains stable on unseen data. The material differences are concentrated in regime composition and downstream inventory-policy metrics, rather than showing evidence of broad model drift.

**Status: LOCKED.**

Saved: `fold3_stability_check.parquet`.


## Section 7 — Final Headline Numbers

This section is a readout of the already-computed Fold 3 results from Section 5b. **No new simulation, model inference, or parameter fitting occurs here.**

### Primary Headline

The locked periodic-review dynamic policy is compared against the static policy at **its own cost-minimizing tested service level** across the 24 real-cost scenarios.

Report:

* **Dynamic vs. static-optimal:** **X%–Y%**
* **Dynamic vs. naive:** **A%–B%**
* **Default scenario (2.0× stockout / 25% carrying):** **$Z estimated annual savings vs. naive**

The static service level shown for each scenario is the **best point within the tested q50–q99 range**, not a confirmed theoretical optimum.

### Supporting Results

At the default q80 configuration:

* Forecast accuracy remained stable relative to Fold 2.
* Dynamic and naive fill/stockout results are reported on the same Fold 3 validation population.
* WAPE remains the primary forecast-side metric for the forecastable Smooth/Erratic regimes.

### Interpretation

The main result is whether the inventory decision layer produces a materially lower modeled cost than the traditional static baseline across a broad range of plausible stockout and carrying-cost assumptions.

These are **Fold 3 holdout results**, so they are reported as final production evidence rather than used to tune the system.

**No changes are made to the pipeline based on these results.**


In [26]:
# -- 07 Section 7: Final Headline Numbers ----------------------------------------------
# Readout only. Uses artifacts already produced in Section 5b.
# No new simulation, model inference, calibration, or parameter selection occurs here.

DEFAULT_MULT = 2.0
DEFAULT_CARRY = 0.25

# ── Required Section 5b artifacts -------------------------------------------------------

assert 'dynamic_sensitivity_df_fold3' in globals(), (
    'dynamic_sensitivity_df_fold3 is missing -- Section 5b must run first.'
)

assert 'headtohead_df_fold3' in globals(), (
    'headtohead_df_fold3 is missing -- Section 5b must run first.'
)

assert 'full_summary_df_fold3' in globals(), (
    'full_summary_df_fold3 is missing -- Section 5 must run first.'
)

assert 'regime_summary_fold3' in globals(), (
    'regime_summary_fold3 is missing -- Section 5 must run first.'
)

# ── Primary headline: dynamic vs static-optimal -----------------------------------------

dynamic_vs_static_optimal = (
    headtohead_df_fold3['dynamic_vs_static_optimal_%']
)

dyn_static_min = dynamic_vs_static_optimal.min()
dyn_static_max = dynamic_vs_static_optimal.max()

# ── Primary headline: dynamic vs naive --------------------------------------------------

dynamic_vs_naive = (
    dynamic_sensitivity_df_fold3['dynamic_vs_naive_%']
)

dyn_naive_min = dynamic_vs_naive.min()
dyn_naive_max = dynamic_vs_naive.max()

# ── Default 2.0x / 25% scenario ----------------------------------------------------------

default_cost_row = dynamic_sensitivity_df_fold3[
    (dynamic_sensitivity_df_fold3['stockout_mult'] == DEFAULT_MULT) &
    (dynamic_sensitivity_df_fold3['carry_rate'] == DEFAULT_CARRY)
].iloc[0]

default_headtohead_row = headtohead_df_fold3[
    (headtohead_df_fold3['stockout_mult'] == DEFAULT_MULT) &
    (headtohead_df_fold3['carry_rate'] == DEFAULT_CARRY)
].iloc[0]

default_naive_cost = default_cost_row['naive_cost_$']
default_dynamic_cost = default_cost_row['dynamic_cost_$']
default_savings = default_naive_cost - default_dynamic_cost
default_dynamic_vs_naive = default_cost_row['dynamic_vs_naive_%']
default_dynamic_vs_static = default_headtohead_row[
    'dynamic_vs_static_optimal_%'
]
default_static_level = default_headtohead_row[
    'static_optimal_level'
]

# ── Supporting q80 metrics ---------------------------------------------------------------
# Section 5's dynamic policy uses the locked q80 safety buffer.
q80_naive_fill = full_summary_df_fold3['naive_fill'].mean()
q80_dynamic_fill = full_summary_df_fold3['dynamic_fill'].mean()

q80_naive_stockout = full_summary_df_fold3[
    'naive_stockout_rate'
].mean()

q80_dynamic_stockout = full_summary_df_fold3[
    'dynamic_stockout_rate'
].mean()

q80_naive_inventory = full_summary_df_fold3[
    'naive_avg_inventory'
].mean()

q80_dynamic_inventory = full_summary_df_fold3[
    'dynamic_avg_inventory'
].mean()

# ── Forecast-side supporting metrics -----------------------------------------------------
forecastable_regimes = ['smooth', 'erratic', 'intermittent']

forecast_summary = regime_summary_fold3.loc[
    regime_summary_fold3['regime'].isin(forecastable_regimes),
    [
        'regime',
        'median_wape',
        'p90_wape',
    ]
].copy()

# ── Static service-level frequency -------------------------------------------------------
static_level_counts = (
    headtohead_df_fold3['static_optimal_level']
    .value_counts()
    .to_dict()
)

# ── Print final readout ------------------------------------------------------------------

print('═' * 88)
print('FOLD 3 — FINAL HEADLINE NUMBERS')
print('═' * 88)

print('\n── Primary headline ──')
print(
    f'Dynamic vs static-optimal: '
    f'{dyn_static_min:.1f}%–{dyn_static_max:.1f}% '
    f'across {len(headtohead_df_fold3):,} cost scenarios'
)

print(
    f'Dynamic vs naive: '
    f'{dyn_naive_min:.1f}%–{dyn_naive_max:.1f}% '
    f'across {len(dynamic_sensitivity_df_fold3):,} cost scenarios'
)

print('\n── Default scenario: 2.0x stockout / 25% carrying ──')
print(
    f'Naive annual cost:      ${default_naive_cost:,.0f}'
)
print(
    f'Dynamic annual cost:    ${default_dynamic_cost:,.0f}'
)
print(
    f'Estimated annual savings: '
    f'${default_savings:,.0f}'
)
print(
    f'Dynamic vs naive:       '
    f'{default_dynamic_vs_naive:.1f}%'
)
print(
    f'Dynamic vs static-opt:  '
    f'{default_dynamic_vs_static:.1f}%'
)
print(
    f'Static tested optimum:  '
    f'{default_static_level}'
)

print('\n── Static tested optimum across scenarios ──')
print(static_level_counts)

print('\n── q80 inventory outcomes ──')
print(
    f'Naive fill rate:         {q80_naive_fill:.3f}'
)
print(
    f'Dynamic fill rate:       {q80_dynamic_fill:.3f}'
)
print(
    f'Naive stockout rate:     {q80_naive_stockout:.3f}'
)
print(
    f'Dynamic stockout rate:   {q80_dynamic_stockout:.3f}'
)
print(
    f'Naive avg inventory:     {q80_naive_inventory:.3f}'
)
print(
    f'Dynamic avg inventory:   {q80_dynamic_inventory:.3f}'
)

print('\n── Forecast accuracy supporting metrics ──')
print(
    forecast_summary.to_string(index=False)
)

print('\n✓ Section 7 complete. No new Fold 3 computation was performed.')

════════════════════════════════════════════════════════════════════════════════════════
FOLD 3 — FINAL HEADLINE NUMBERS
════════════════════════════════════════════════════════════════════════════════════════

── Primary headline ──
Dynamic vs static-optimal: 87.5%–91.8% across 24 cost scenarios
Dynamic vs naive: 92.7%–95.4% across 24 cost scenarios

── Default scenario: 2.0x stockout / 25% carrying ──
Naive annual cost:      $32,140,627
Dynamic annual cost:    $1,989,151
Estimated annual savings: $30,151,476
Dynamic vs naive:       93.8%
Dynamic vs static-opt:  89.2%
Static tested optimum:  q99

── Static tested optimum across scenarios ──
{'q99': 24}

── q80 inventory outcomes ──
Naive fill rate:         0.589
Dynamic fill rate:       0.975
Naive stockout rate:     0.411
Dynamic stockout rate:   0.025
Naive avg inventory:     4.673
Dynamic avg inventory:   24.598

── Forecast accuracy supporting metrics ──
      regime  median_wape  p90_wape
     erratic        0.425     0.747
inter

# 7. Final Headline Numbers

Fold 3 confirms the production policy under the frozen model, regime, calibration, and inventory-policy configuration.

## Primary cost results

Across 24 real-cost scenarios:

* **Dynamic vs. static-optimal:** **87.5%–91.8% lower estimated annual cost**
* **Dynamic vs. naive:** **92.7%–95.4% lower estimated annual cost**
* Static inventory selects **q99 in all 24 scenarios**; this is the best tested static level, not evidence of a global optimum outside the tested range.

### Default cost scenario

Using the default **2.0× stockout penalty / 25% annual carrying rate**:

| Policy            | Estimated annual cost |
| ----------------- | --------------------: |
| Naive             |           $32,140,627 |
| Dynamic           |            $1,989,151 |
| Estimated savings |       **$30,151,476** |

At this scenario, the dynamic policy produces:

* **93.8% lower cost than naive**
* **89.2% lower cost than the static tested optimum**
* Static tested optimum: **q99**

These results are cost comparisons under the specified stockout and carrying-cost assumptions; they should not be interpreted as equal-inventory comparisons.

## Service and inventory outcomes

At the locked **q80** dynamic service level:

| Metric            | Naive |    Dynamic |
| ----------------- | ----: | ---------: |
| Fill rate         | 0.589 |  **0.975** |
| Stockout rate     | 0.411 |  **0.025** |
| Average inventory | 4.673 | **24.598** |

The dynamic policy therefore substantially improves service, while carrying more inventory. The primary justification for the policy is the resulting **total-cost reduction**, rather than a reduction in inventory alone.

## Forecast accuracy

Per-SKU forecast accuracy remains broadly stable relative to Fold 2:

| Regime       | Median WAPE | p90 WAPE |
| ------------ | ----------: | -------: |
| Smooth       |   **0.365** |    0.646 |
| Erratic      |   **0.425** |    0.747 |
| Intermittent |   **0.879** |    1.546 |

The relatively high intermittent-regime WAPE is consistent with Fold 2 and reflects the difficulty of forecasting sparse, intermittent demand. Fold 3 does not show a deterioration in forecast accuracy: median WAPE improved slightly for Smooth and Intermittent SKUs and was unchanged for Erratic SKUs.

## Final interpretation

Fold 3 provides a one-shot final evaluation of the locked production pipeline. Forecast accuracy is stable across folds, while the inventory simulation shows substantial cost advantages for the dynamic policy across the tested cost scenarios. The results are reported without additional tuning or parameter selection after observing Fold 3.


In [27]:
# Save canonical Fold 3 ML forecast artifact for downstream notebooks.
final_predictions_fold3 = traj_df_fold3[
    ['id', 'date', 'yhat']
].copy()

final_predictions_fold3.to_parquet(
    f'{PREDICTIONS_DIR}/final_predictions_fold3.parquet',
    index=False
)

print(
    f"Saved: {PREDICTIONS_DIR}/final_predictions_fold3.parquet "
    f"({len(final_predictions_fold3):,} rows, "
    f"{final_predictions_fold3['id'].nunique():,} SKUs)"
)

Saved: ../data/processed/predictions/final_predictions_fold3.parquet (3,234,995 rows, 8,863 SKUs)


# 8. Final Signoff

Fold 3 is the final one-shot evaluation of the locked forecasting and inventory-policy pipeline.

The production configuration is frozen as:

* **Forecast model:** LightGBM Tweedie, raw variant
* **SKU routing:** Syntetos-Boylan demand regimes
* **Smooth / Erratic:** Tweedie forecast with locked q80 service level
* **Intermittent:** TSB-based policy
* **Lumpy:** historical inventory policy
* **Conformal calibration:** Fold 3 monitor-window calibration with frozen service levels
* **No post-Fold-3 tuning or parameter selection**

The final Fold 3 results show:

* **87.5%–91.8% lower estimated annual cost vs. static-optimal** across 24 tested cost scenarios
* **92.7%–95.4% lower estimated annual cost vs. naive**
* **93.8% lower cost vs. naive** in the default 2.0× stockout / 25% carrying-cost scenario
* **97.5% dynamic fill rate** at the locked q80 policy
* Forecast accuracy remained broadly stable relative to Fold 2

These results are considered the final evaluation for the current project version. The model, routing logic, calibration, and inventory-policy parameters are now **locked**. Any future changes should be treated as a new experiment and evaluated on a new holdout rather than retuned against Fold 3.

### Final Artifacts

The following artifacts constitute the frozen Fold 3 output:

* `tweedie_optimized_fold3.txt`
* `sku_regimes_fold3.parquet`
* `conformal_residuals_fold3.pkl`
* `final_predictions_fold3.parquet`
* `dynamic_cost_sensitivity_fold3.parquet`
* `static_vs_dynamic_headtohead_fold3.parquet`
* `fold3_stability_check.parquet`
